The SQL query pulls in all the data from the All of Us dataset. The resulting data table will have each person with their person ID, the date of the event, and the ICD code that corresponds to their visit.

# Filter Records and set relative to DayZero times

In [ ]:
import pandas as pd
import os
import numpy as np
from datetime import *

dataset = %env WORKSPACE_CDR
dataset

query = f"""
SELECT person_id, condition_start_datetime, condition_source_value, source_concept_name as icd_description
FROM `{dataset}.ds_condition_occurrence` co
WHERE co.PERSON_ID IN (
    SELECT
        distinct person_id
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person
    WHERE
        cb_search_person.person_id IN (
            SELECT
                person_id
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p
        )
    )
"""

all_data = pd.read_gbq(query,
                        dialect = "standard",
                        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
                        progress_bar_type="tqdm_notebook")

all_data.head()

# Add concept_code from concept OMOP table

In [ ]:
import pandas as pd
import os
import numpy as np
from datetime import *

dataset = %env WORKSPACE_CDR
dataset

query = f"""
SELECT
    person_id,
    condition_start_datetime,
    condition_source_value,
    source_concept_name as icd_description,
    concept_code,
    c.concept_id
FROM
    `{dataset}.ds_condition_occurrence` co
LEFT JOIN
    `{dataset}.concept` c ON co.condition_concept_id = c.concept_id
WHERE co.PERSON_ID IN
    (
        SELECT
            distinct person_id
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person
        WHERE
            cb_search_person.person_id IN

            (
                SELECT
                    person_id
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p
            )
    )
"""

all_codes = pd.read_gbq(query,
                        dialect = "standard",
                        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
                        progress_bar_type="tqdm_notebook")

all_codes.head()

In [ ]:
import pandas as pd
import os
import numpy as np
from datetime import *

dataset = %env WORKSPACE_CDR
dataset

query = f"""
SELECT person_id, condition_start_datetime, condition_source_value, source_concept_name as icd_description
FROM `{dataset}.ds_condition_occurrence` co
WHERE co.PERSON_ID IN (
    SELECT
        distinct person_id
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person
    WHERE
        cb_search_person.person_id IN (
            SELECT
                person_id
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p
        )
    )
    AND (UPPER(source_concept_name) LIKE '%MALIGNANT%' AND UPPER(source_concept_name) LIKE '%NEOPLASM%')
    AND (condition_source_value LIKE 'C50%' OR condition_source_value LIKE '174%')
"""

malignant_data = pd.read_gbq(query,
                        dialect = "standard",
                        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
                        progress_bar_type="tqdm_notebook")

malignant_data.head()

The following code uses the function get_descriptions() to create a new column in the data frame with the description that corresponds to the ICD code

To narrow the data pool, we search for all events that are tied to "malignant neoplasm" and then order the data by person_id, and by event date.

In [ ]:
unique_count = malignant_data['person_id'].nunique()
print(f'There are {unique_count} total people in the AoU dataset')

In [ ]:
print(malignant_data["condition_source_value"])

In [ ]:
cancer_sorted = malignant_data.sort_values(by='person_id').groupby('person_id', group_keys=False).apply(lambda x: x.sort_values(by='condition_start_datetime'))
cancer_sorted = cancer_sorted.reset_index(drop=True)
cancer_sorted.head()

Using the numpy library, we filter the data from the cancer_sorted DataFrame to create a new data table that has the first and last event dates tied to each person. The code then filters out whoever does not have multiple events.

In [ ]:
minmax_cancer_data = cancer_sorted.groupby(['person_id']).agg({'condition_start_datetime': [np.min,np.max]})
minmax_cancer_data.columns = minmax_cancer_data.columns.get_level_values(1)
minmax_cancer_data = minmax_cancer_data.reset_index()
minmax_cancer_data.head()

In [ ]:
new_minmax_data = pd.DataFrame()
for index, row in minmax_cancer_data.iterrows():
    if row['amin'] != row['amax']:
        row_to_extract = minmax_cancer_data.iloc[index]
        new_minmax_data = pd.concat([new_minmax_data, pd.DataFrame([row_to_extract])])
new_minmax_data = new_minmax_data.reset_index(drop=True)
new_minmax_data.head()
print(f'we have {len(new_minmax_data)} people')

We take the new verified_cancer_data table, and the 'min' date from the final Min/Max DataFrame to find how far apart each event for each person is. The min date from the Min/Max table acts as 'day0' or the first occurance of the malignant neoplasm.

The following code, creates the dayzero column, but for all the events in a person's timeline who has a 'malignant neoplasm' in their records

In [ ]:
beforeandafter = pd.DataFrame()
shared_data = all_codes['person_id'].isin(new_minmax_data['person_id'])
beforeandafter = all_codes[shared_data]
beforeandafter.head()

In [ ]:
banda_sorted = beforeandafter.sort_values(by='person_id').groupby('person_id', group_keys=False).apply(lambda x: x.sort_values(by='condition_start_datetime'))
banda_sorted = banda_sorted.reset_index(drop=True)
banda_sorted.head()

In [ ]:
# Group new_minmax_data by 'person_id' and select the 'amin' value for each person
amin_mapping = new_minmax_data.groupby('person_id')['amin'].first()

# Create 'day0_relative_time' by mapping 'person_id' to the corresponding 'amin' value
banda_sorted['day0_relative_time'] = banda_sorted['person_id'].map(amin_mapping)

# Calculate 'day_diff' using the mapping and vectorized operations
banda_sorted['day_diff'] = (banda_sorted['condition_start_datetime'] - banda_sorted['day0_relative_time']).dt.days
del banda_sorted['day0_relative_time']
banda_sorted.head()

The following code works, but it is really slow. Use the above code to find the day_diff column more efficiently.

BIN CODE

In [ ]:
# uses vectorized function to filter negative days

neg_days = banda_sorted[banda_sorted['day_diff'] <= 0]

neg_days.head()

only run the cell below if you want to read the codes into bitstring (possibly helpful for wdl)

In [ ]:
#TRANSLATION OF THE CONCEPT ID STRINGS INTO BITSTRING
#DICTIONARY REF: conceptIDDictionary

#function that changes string into bitString
def string_to_bitstring(input_string):
    byte_string = input_string.encode('utf-8')

    bitstring = ''.join(format(byte, '08b') for byte in byte_string)

    return bitstring

#gets all of the unique concept IDs in the dataframe

uniquePersonIDs = list(neg_days['icd_description'].unique())

#filters out any NoneType

uniquePersonIDs = [ID for ID in uniquePersonIDs if ID is not None]

conceptIDDictionary = {}

#length of list is 30944, datatype is all string

#pairs StringID as key and bitString as value

for ID in uniquePersonIDs:
    bitString = string_to_bitstring(ID)
    conceptIDDictionary[ID] = bitString

# Iterate over each row in the DataFrame
for index, row in neg_days.iterrows():
    # Get the original string from the 'icd_description' column
    original_string = row['icd_description']
    if original_string is not None:

        # Translate the original string into a bitstring
        bitstring = conceptIDDictionary[original_string]

        # Update the 'icd_description' column with the bitstring version
        neg_days.at[index, 'icd_description'] = bitstring

neg_days.head()

In [ ]:
first_events = neg_days.groupby('person_id')['day_diff'].min().reset_index()
print(len(first_events))
first_events.head()

In [ ]:
bins = np.arange(-15000, 100, 100)
labels = [f"{bins[i]},{bins[i+1]}" for i in range(len(bins)-1)]
print(bins)
print(labels)

In [ ]:
neg_days['bin'] = pd.cut(neg_days['day_diff'],labels=labels,bins=bins)
binned_cancer_data = neg_days
binned_cancer_data.head(30)

In [ ]:
# Bins each event
neg_days['bin'] = pd.cut(neg_days['day_diff'],labels=labels,bins=bins)
neg_days.head()

value_counts = neg_days['bin'].value_counts().sort_index().reset_index()
value_counts.head(10)
num_events = value_counts['count'].tolist()
print(num_events)

binned_individuals = binned_cancer_data[pd.notna(binned_cancer_data['bin'])]
binned_individuals.head(30)

# UPLOADING binned_individuals into the Google Workspace Bucket

In [ ]:
# SETUP

import os
import subprocess
import numpy as np
import pandas as pd

In [ ]:
# Check if df exists
binned_individuals.head()

In [ ]:
# This snippet assumes you run setup first
# TAKES A WHILE (10-15 minutes) but as as long as the disk is not deleted we have complete access to our pre-filtered data.
# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
binned_individuals['condition_start_datetime'] = binned_individuals['condition_start_datetime'].astype(str)

my_dataframe = binned_individuals

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename = 'binned_individuals.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# save dataframe in a csv file in the same workspace as the notebook
my_dataframe.to_csv(destination_filename, index=False)

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file to the bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/data/"]
output = subprocess.run(args, capture_output=True)

# print output from gsutil
output.stderr

In [ ]:
import json

groupedDataFrame = binned_individuals.groupby('person_id')
dfArray = [json.dumps(group.reset_index(drop=True).to_dict()) for _, group in groupedDataFrame]

unique_personIDs = list(binned_individuals['person_id'].unique())
# print(unique_personIDs)

In [ ]:
for i in range(len(unique_personIDs)):
    my_dataframe = binned_individuals
    destination_filename = f"input{i}.txt"
    with open(destination_filename, 'w') as output:
        output.write(dfArray[i])
my_bucket = os.getenv('WORKSPACE_BUCKET')
args = ["gsutil","-m", "cp", "./input*.txt", f"{my_bucket}/data/"]
subprocess.run(args, capture_output=False)

# RUN FROM HERE

This code assumes that the file is still uploaded in the google workspace bucket. Run the cells above if this fails. As long as we don't delete the disk attatched to this workspace, we have full access to it. Establishes binned_individuals as the main DataFrame

In [ ]:
# first argument is how many days back our bins should go, the second is the size of each bin
# returns two lists, the first is the cuts of each bin and the second is the name of the respective bins
def bins_n_labels(time_range, bin_size):
    bins = []
    labels = []
    n = -time_range
    while n <= 0:
        bins.append(n)
        label = f'{n},'
        n += bin_size
        label += f'{n}'
        labels.append(label)
    return bins, labels[:-1]

bins, labels = bins_n_labels(15000,100)
# arg 1 is how far back you want to look, arg 2 is the size of each bin you want.
print(bins)
# print(bins)
print(len(bins))
# print(labels)
print(len(labels))
# neg_days['bins'] = pd.cut(neg_days['values'], bins=bins, labels=labels)

In [ ]:
# SETUP

import os
import subprocess
import numpy as np
import pandas as pd


# This snippet assumes you run setup first
# Get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')
# List objects in the bucket
# print(subprocess.check_output(f"gsutil ls -r {my_bucket}", shell=True).decode('utf-8'))
# This code copies file in your Google Bucket and loads it into a dataframe

# Replace 'test.csv' with THE NAME of the file you're going to download from the bucket (don't delete the quotation marks)
name_of_file_in_bucket = 'binned_individuals.csv'

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

# print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
binned_individuals = pd.read_csv(name_of_file_in_bucket)
binned_individuals.head()

In [ ]:
# Check if df exists
binned_individuals.head()

Below is a function that grabs an individuals records from the Google Workspace files. It takes in an integer person_id, then returns a DataFrame of that persons pre-cancer data.

In [ ]:
def get_individual_records(id, unique_personIDs):
    # This snippet assumes you run setup first

    # This code copies file in your Google Bucket and loads it into a dataframe
    # Replace 'test.csv' with THE NAME of the file you're going to download from the bucket (don't delete the quotation marks)
    name_of_file_in_bucket = f"input{unique_personIDs.index(id)}.txt"
    my_bucket = os.getenv('WORKSPACE_BUCKET')

    os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

    print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
    # save dataframe in a csv file in the same workspace as the notebook

    with open(name_of_file_in_bucket) as infile:
        json_data = infile.readline()
    return pd.DataFrame(eval(json_data))

In [ ]:
# test to see if it worked

individual_df = get_individual_records(1000066, unique_personIDs)
individual_df.head()

In [ ]:
# This snippet assumes that you run setup first

# This code lists objects in your Google Bucket

# Get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# List objects in the bucket
print(subprocess.check_output(f"gsutil ls -r {my_bucket}", shell=True).decode('utf-8'))

In [ ]:
# This snippet assumes that you run setup first

# This code lists objects in your Google Bucket

# Get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# List objects in the bucket
print(subprocess.check_output(f"gsutil ls -r {my_bucket}", shell=True).decode('utf-8'))

# Nextflow

In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd

# Create and Write Dataframe to Workspace Bucket

First we need to grab the proper sample size of individuals we want to test. Remember, these are numbers of individuals, not comparisons

In [ ]:
# NUMBER OF PEOPLE WE'D LIKE TO SAMPLE
n_people = 10

In [ ]:
random_ids = binned_individuals['person_id'].sample(n=n_people).tolist()
random_ids

sample_df = binned_individuals[binned_individuals['person_id'].isin(random_ids)]
sample_df

In [ ]:
# This snippet assumes you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe = sample_df

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename = 'sample_workflow_input.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# save dataframe in a csv file in the same workspace as the notebook
my_dataframe.to_csv(destination_filename, index=False)

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file to the bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/data/"]
output = subprocess.run(args, capture_output=True)

# print output from gsutil
output.stderr

In [ ]:
sample_df.columns

# Setting up Nextflow

Here we initialize nexflow and check that it is installed

In [ ]:
os.environ['PET_SA_EMAIL'] = subprocess.check_output(["gcloud", "config", "get-value", "account"]).decode("utf-8").strip()
!nextflow

In [ ]:
!echo $PET_SA_EMAIL

In [ ]:
my_bucket = os.getenv('WORKSPACE_BUCKET')
my_bucket

In [ ]:
# %%writefile parse_input.py

# import sys
# import argparse

# def parse_list(arg_string):
#     # Remove brackets and split by comma
#     cleaned = arg_string.strip('[]')
#     return [item.strip() for item in cleaned.split(',')]

# parser = argparse.ArgumentParser(description='Process EHR data')
# parser.add_argument('person_id', type=str, help='Person ID')
# parser.add_argument('condition_start_datetime', type=parse_list, help='List of condition start datetimes')
# parser.add_argument('condition_source_value', type=parse_list, help='List of condition source values')
# parser.add_argument('icd_description', type=parse_list, help='List of ICD descriptions')
# parser.add_argument('concept_code', type=parse_list, help='List of concept codes')
# parser.add_argument('concept_id', type=parse_list, help='List of concept IDs')
# parser.add_argument('day_diff', type=parse_list, help='List of day differences')
# parser.add_argument('bin', type=parse_list, help='List of bins')

# args = parser.parse_args()

# person_id = args.person_id
# condition_start_datetime = args.condition_start_datetime
# condition_source_value = args.condition_source_value
# icd_description = args.icd_description
# concept_code = args.concept_code
# concept_id = args.concept_id
# day_diff = args.day_diff
# bin_list = args.bin

# # Check if all lists have the same length
# if not (len(condition_start_datetime) == len(condition_source_value) == len(icd_description) == len(concept_code) == len(concept_id) == len(day_diff) == len(bin_list)):
#     print("Error: All input lists must have the same length.")
#     sys.exit(1)

# with open(f"person_{person_id}.csv","w") as out:
#     # Write the header
#     out.write("person_id,condition_start_datetime,condition_source_value,icd_description,concept_code,concept_id,day_diff,bin\n")
#     # Write each record
#     for datetime, source_value, description, code, id, diff, bin_value in zip(condition_start_datetime, condition_source_value, icd_description, concept_code, concept_id, day_diff, bin_list):
#         out.write(",".join([person_id, datetime, source_value, description, code, id, diff, bin_value]) + "\n")

#     out.write("\n\n")

In [ ]:
%%writefile jaccard_scorer.py


# #UPDATED: THIS CELL IS THE MOST RECENT VERSION
# from collections import OrderedDict
# import statistics as st
# import numpy as np
# import itertools
# import copy
# import sys
# import pandas as pd

# person1_df = pd.read_csv(sys.argv[1])
# person2_df = pd.read_csv(sys.argv[2])


# rando1 = person1_df.iloc[0,0]
# rando2 = person2_df.iloc[0,0]
# filename = f"person_{rando1}_person_{rando2}.txt"
# with open(filename, "w") as out:
#     out.write(f"{rando1},{rando2}")



#UPDATED: THIS CELL IS THE MOST RECENT VERSION
import pandas as pd
from collections import OrderedDict
import statistics as st
import numpy as np
import sys

multiple_links = []
indexes2actuallyplot = []
jscore_to_bin_dict = {}
matching_codes = []
# hypertension_bins = []
# hyperlipidemia_bins = []
# e_hypertension_bins = []
# hypercholesterolema_bins = []
# diabetes_bins = []
# benign_bins = []
people_compared = {}
people_bins_avgScores = {}
averages_with_bins = {}
split_avg = {}

person1_df = pd.read_csv(sys.argv[1])
person2_df = pd.read_csv(sys.argv[2])


rando1 = person1_df.iloc[0,0]
rando2 = person2_df.iloc[0,0]

# Filter DataFrame for rows belonging to person 1

# Extract the 'bin' column values into a list
bin_values_person1 = person1_df['bin'].tolist()

person1_bin_set = set(bin_values_person1)

# Filter DataFrame for rows belonging to person 2
# Extract the 'bin' column values into a list
bin_values_person2 = person2_df['bin'].tolist()
person2_bin_set = set(bin_values_person2)

similarity_scores_matrix = []
bins_compared = []

# possibly use iter tools for the combinations
# use itertools to get the bin combinations to eliminate the double 'for' loop


for bin1 in person1_bin_set:
    bin_scores = []
    bin_comparisons = []
    for bin2 in person2_bin_set:
        Asub = set(person1_df.loc[(person1_df['bin'] == bin1), 'concept_code'])
        Bsub = set(person2_df.loc[(person2_df['bin'] == bin2), 'concept_code'])
        C = Asub.intersection(Bsub)
        D = Asub.union(Bsub)
        if len(D) == 0:
            jsim = 0.0
        else:
            jsim = float(len(C))/float(len(D))

        if len(C) > 0:
            matching_codes.append(C)
#                 for item in C:

#                     # search for benign ICDs (D for ICD 10, 2+ 1 or 2 for ICD 9)
#                     if item[0] == 'D':
#                         benign_bins.append(bin1)
#                         benign_bins.append(bin2)
#                     elif item[0] == '2':
#                         if item[1] == '1':
#                             benign_bins.append(bin1)
#                             benign_bins.append(bin2)
#                         elif item[1] == '2':
#                             benign_bins.append(bin1)
#                             benign_bins.append(bin2)

#                     # searching for conditions in ICD 9
#                     if item == '401.9':
#                         hypertension_bins.append(bin1)
#                         hypertension_bins.append(bin2)
#                     elif item == '272.4':
#                         hyperlipidemia_bins.append(bin1)
#                         hyperlipidemia_bins.append(bin2)

#                     #searching for conditions in ICD 10
#                     elif item == 'I10':
#                         e_hypertension_bins.append(bin1)
#                         e_hypertension_bins.append(bin2)

#                     #ICD 9

#                     elif item == '272.0':
#                         hypercholesterolema_bins.append(bin1)
#                         hypercholesterolema_bins.append(bin2)
#                     elif item == '250.00':
#                         diabetes_bins.append(bin1)
#                         diabetes_bins.append(bin2)

        # search for significant similarity in Jscore (significance arb set as 0.1 for now)
#             people_compared.setdefault((rando1,rando2),[])

        # for each pair of people compared, we are putting in the score for each of their bin comparisons
#             people_compared[(rando1, rando2)].append({(bin1, bin2) : jsim})

        bin_scores.append(jsim)
        bin_comparisons.append((bin1, bin2))
        jscore_to_bin_dict[jsim] = (bin1, bin2)
    similarity_scores_matrix.append(bin_scores)
    bins_compared.append(bin_comparisons)


#row indexes correspond to indexes of person1_bin_set, column indexes correspond to person2_bin_set indexes

#part 2:
max_value = 0.0
max_row = 0
max_column = 0
exclude_rows = []
exclude_columns = []
max_links = []
maxes = []
excluded_bins = []
max_dict_ordered = OrderedDict()
excluded_bins_to_raw_score = OrderedDict()


similarity_scores_matrix_array = np.array(similarity_scores_matrix)

similarity_matrix = np.asmatrix(similarity_scores_matrix_array)

# goal is to make sure that the two bins compared are associated with the best score

bins_compared = np.array(bins_compared)

while len(similarity_scores_matrix_array)>0:

    max_index = np.argmax(similarity_scores_matrix_array)

    max_score = similarity_matrix.max()

    max_row, max_col = np.unravel_index(max_index, similarity_scores_matrix_array.shape)

    exclude_rows.append(max_row)
    exclude_columns.append(max_col)
    maxes.append(max_score)

    tupleKey = (bins_compared[max_row][max_col][0],bins_compared[max_row][max_col][1])

    excluded_bins_to_raw_score.setdefault(tupleKey,[])
    excluded_bins_to_raw_score[tupleKey].append(similarity_scores_matrix_array[max_row][max_col])

    max_dict_ordered[(max_row, max_col)] = max_score

    similarity_scores_matrix_array = np.delete(similarity_scores_matrix_array,max_row,0)
    similarity_scores_matrix_array = np.delete(similarity_scores_matrix_array,max_col,1)

    bins_compared = np.delete(bins_compared,max_row, 0)
    bins_compared = np.delete(bins_compared,max_col, 1)

    similarity_matrix = np.delete(similarity_matrix,max_row, 0)
    similarity_matrix = np.delete(similarity_matrix, max_col, 1)

    if similarity_scores_matrix_array.size == 0:
        break

#part 4
# Sort the keys
sorted_keys = sorted(max_dict_ordered.keys(), reverse=True)

# Create a new ordered dictionary with sorted keys
sorted_ordered_dict = OrderedDict((key, max_dict_ordered[key]) for key in sorted_keys)
links_to_graph = []
for key1, value in sorted_ordered_dict.items():
    links_to_graph.append(value)

averages_to_actually_graph = []
# introduce an expand
# eliminate
# x is the bin we took out, and y is the average score

while sorted_ordered_dict:
    avg_2_plot = [sorted_ordered_dict[key] for key in sorted_ordered_dict]
    averages_to_actually_graph.append(st.mean(avg_2_plot))
    indexes2use = [key for key in sorted_ordered_dict]
    indexes2actuallyplot.append(indexes2use)
    sorted_ordered_dict.popitem()

sorted_keys = sorted(excluded_bins_to_raw_score.keys(), reverse=True)
sorted_bins_to_max_score_dict = OrderedDict((key, excluded_bins_to_raw_score[key]) for key in sorted_keys)
while sorted_bins_to_max_score_dict:
    maxes_from_matrix = [item for key in sorted_bins_to_max_score_dict for item in sorted_bins_to_max_score_dict[key]]
    bins_from_matrix = [key for key in sorted_bins_to_max_score_dict]
    maxes_average = st.mean(maxes_from_matrix)
    if (rando1, rando2) not in averages_with_bins:
        averages_with_bins[(rando1, rando2)] = []
    averages_with_bins[(rando1, rando2)].append({bins_from_matrix[-1] : maxes_average})
    sorted_bins_to_max_score_dict.popitem()
person1_best_scores = {}
person2_best_scores = {}
#splitting the dictionary
for key, value in excluded_bins_to_raw_score.items():
    person1_best_scores[key[0]] = value[0]
    person2_best_scores[key[1]] = value[0]
sorted_by_bins_person1 = dict(sorted(person1_best_scores.items()))
sorted_by_bins_person2 = dict(sorted(person2_best_scores.items()))
if (rando1, rando2) not in split_avg:
    split_avg[(rando1, rando2)] = []
curr_average = 0
while sorted_by_bins_person2:
    closest_to_diagnosis_p2 = min(sorted_by_bins_person2)
    values = sorted_by_bins_person2.values()
    total = sum(values)
    curr_average = total / len(values)
    split_avg[(rando1, rando2)].append({closest_to_diagnosis_p2:curr_average})
    sorted_by_bins_person2.pop(closest_to_diagnosis_p2)

new_averages = []
y_intercept = averages_to_actually_graph[0]
for item in averages_to_actually_graph:
    new_averages.append(item - y_intercept)
multiple_links.append(new_averages)


filename = f"person_{rando1}_person_{rando2}.txt"
with open(filename, "w") as out:
    out.write(f"{split_avg}")

In [ ]:
# workflow {
#     // Read and process data
#     Channel.fromPath(params.input_csv)
#         .splitCsv(header: true)
#         .map { row -> tuple(row["Person ID"], row) }
#         .groupTuple()
#         .map { person_id, rows ->
#             def header = "person_id,condition_start_datetime,condition_source_value,cd_description,concept_code,concept_id,day_diff,bin"
#             def content = rows.collect { r ->
#                 "${r['person_id']},${r['condition_start_datetime']},${r['condition_source_value']},${r['cd_description']},${r['concept_code']},${r['concept_id']},${r['day_diff']},${r['bin']}"
#             }.join('\n')
#             // Create file object with explicit path
#             def csv_file = file("${params.output_dir}/person_${person_id}.csv")
#             csv_file.text = "${header}\n${content}"
#             // Verify file creation
#             if (!csv_file.exists()) {
#                 throw new Exception("Failed to create: ${csv_file}")
#             }
#             csv_file
#         }
#         .set { individual_csvs }
#         individual_csvs
#         .combine(individual_csvs)
#         .filter { file1, file2 -> file1.name < file2.name }
#         .set { pairwise_combinations }


#         jaccard_results = jaccardScoring(pairwise_combinations)
#         jaccard_results
#         .collectFile(name: "/workspaces/training/hello-nextflow/windels_workflow/combined_jaccard_results.csv", newLine: true)

#     // Rest of workflow remains the same...
# }

In [ ]:
%%writefile windel_workflow.nf

nextflow.enable.dsl=2

process jaccardScoring {
    input:
    tuple path(file1), path(file2)
    path scorer
    script:
    """
    python ${scorer} $file1 $file2
    """
    output:
    path "${file1.simpleName}_${file2.simpleName}.txt"
}

params.input_csv = "sample_workflow_input.csv"
params.output_dir = "gs://fc-secure-b288caa9-2b55-4ff4-a30a-036c77face90/individual_csvs"


workflow {
    // Create output directory first
    scorer = Channel.fromPath("gs://fc-secure-b288caa9-2b55-4ff4-a30a-036c77face90/jaccard_scorer.py")
    Channel.fromPath(params.input_csv)
        .splitCsv(header: true)
        .map { row ->
            tuple(row["person_id"], row)  // Changed from "Person ID" to match header
        }
        .groupTuple()
        .map { person_id, rows ->
            // Create CSV content with proper field mapping
            def header = "person_id,condition_start_datetime,condition_source_value,cd_description,concept_code,concept_id,day_diff,bin"
            def content = rows.collect { r ->
                "${r.person_id},${r.condition_start_datetime},${r.condition_source_value}," +
                "${r.cd_description},${r.concept_code},${r.concept_id},${r.day_diff},${r.bin}"
            }.join('\n')

            // Create file path using params
            def csv_path = "${params.output_dir}/person_${person_id}.csv"

            // Write to Google Cloud Storage using gsutil
            """
            echo '${header}\n${content}' | gsutil cp - ${csv_path}
            """

            // Return file path for downstream processing
            file(csv_path)
        }
        .set { individual_csvs }

    // Create pairwise combinations
    pairwise_combinations = individual_csvs
        .combine(individual_csvs)
        .filter { file1, file2 -> file1.name < file2.name }

    // Execute jaccard scoring
    jaccardScoring(pairwise_combinations,scorer)
        .collectFile(
            name: "gs://fc-secure-b288caa9-2b55-4ff4-a30a-036c77face90/combined_jaccard_results.csv",
            newLine: true
        )
}

In [ ]:
%%bash
cat <<EOF > ~/.nextflow/config
profiles {
  gls {
      process.executor = "google-lifesciences"
      process.container = ""
      workDir = "gs://fc-secure-b135d9b1-d832-4f2d-90b0-c073a6700872/workflows/nextflow-scratch"
      google.location = "us-central1"
      google.zone = "us-central1-a"
      google.project = "terra-vpc-sc-a4b0d6af"
      google.enableRequesterPaysBuckets = true
      google.lifeSciences.debug = true
      google.lifeSciences.serviceAccountEmail = "pet-2689124224761d9c7c248@terra-vpc-sc-1c3a1dab.iam.gserviceaccount.com"
      google.lifeSciences.network = "network"
      google.lifeSciences.subnetwork = "subnetwork"
      google.lifeSciences.usePrivateAddress = false
      google.lifeSciences.copyImage = "gcr.io/google.com/cloudsdktool/cloud-sdk:alpine"
      google.lifeSciences.bootDiskSize = "20.GB"
  }
}
EOF

TODO:

Move back to ubuntu default image
Test if it reaches dependency issues
Generate a diagnostic .nf file
Check EOF indentation errors

In [ ]:
cat ~/.nextflow/config

In [ ]:
!echo ${ARTIFACT_REGISTRY_DOCKER_REPO}

In [ ]:
!gsutil cp jaccard_scorer.py gs://fc-secure-b288caa9-2b55-4ff4-a30a-036c77face90/
!gsutil cp parse_input.py gs://fc-secure-b288caa9-2b55-4ff4-a30a-036c77face90/

In [ ]:
!nextflow run windel_workflow.nf -c ~/.nextflow/config -profile gls  -process.container="${ARTIFACT_REGISTRY_DOCKER_REPO}/ebang1919/windel:dependencies"

In [ ]:
!gsutil cat gs://fc-secure-b288caa9-2b55-4ff4-a30a-036c77face90/jaccard_scorer.py

# Jaccard Recent Version in below cell

Be sure to use the 4.5 minute version when running comparisons

In [ ]:
#UPDATED: THIS CELL IS THE MOST RECENT VERSION

#2000 comparisons
from collections import OrderedDict
import statistics as st
import numpy as np
import itertools
import copy

multiple_links = []
indexes2actuallyplot = []
jscore_to_bin_dict = {}
matching_codes = []
# hypertension_bins = []
# hyperlipidemia_bins = []
# e_hypertension_bins = []
# hypercholesterolema_bins = []
# diabetes_bins = []
# benign_bins = []
people_compared = {}
people_bins_avgScores = {}
averages_with_bins = {}
split_avg = {}

# for _ in range(2000):
for _ in range(100):
    rando1 = binned_individuals['person_id'].sample().iloc[0]
    rando2 = binned_individuals['person_id'].sample().iloc[0]

    # Filter DataFrame for rows belonging to person 1
    person1_df = binned_individuals[binned_individuals['person_id'] == rando1]

    # Extract the 'bin' column values into a list
    bin_values_person1 = person1_df['bin'].tolist()

    person1_bin_set = set(bin_values_person1)

    # Filter DataFrame for rows belonging to person 2
    person2_df = binned_individuals[binned_individuals['person_id'] == rando2]
    # Extract the 'bin' column values into a list
    bin_values_person2 = person2_df['bin'].tolist()

    person2_bin_set = set(bin_values_person2)

    similarity_scores_matrix = []
    bins_compared = []

    # possibly use iter tools for the combinations
    # use itertools to get the bin combinations to eliminate the double 'for' loop


    for bin1 in person1_bin_set:
        bin_scores = []
        bin_comparisons = []
        for bin2 in person2_bin_set:
            Asub = set(person1_df.loc[(person1_df['bin'] == bin1), 'concept_code'])
            Bsub = set(person2_df.loc[(person2_df['bin'] == bin2), 'concept_code'])
            C = Asub.intersection(Bsub)
            D = Asub.union(Bsub)
            if len(D) == 0:
                jsim = 0.0
            else:
                jsim = float(len(C))/float(len(D))

            if len(C) > 0:
                matching_codes.append(C)
#                 for item in C:

#                     # search for benign ICDs (D for ICD 10, 2+ 1 or 2 for ICD 9)
#                     if item[0] == 'D':
#                         benign_bins.append(bin1)
#                         benign_bins.append(bin2)
#                     elif item[0] == '2':
#                         if item[1] == '1':
#                             benign_bins.append(bin1)
#                             benign_bins.append(bin2)
#                         elif item[1] == '2':
#                             benign_bins.append(bin1)
#                             benign_bins.append(bin2)

#                     # searching for conditions in ICD 9
#                     if item == '401.9':
#                         hypertension_bins.append(bin1)
#                         hypertension_bins.append(bin2)
#                     elif item == '272.4':
#                         hyperlipidemia_bins.append(bin1)
#                         hyperlipidemia_bins.append(bin2)

#                     #searching for conditions in ICD 10
#                     elif item == 'I10':
#                         e_hypertension_bins.append(bin1)
#                         e_hypertension_bins.append(bin2)

#                     #ICD 9

#                     elif item == '272.0':
#                         hypercholesterolema_bins.append(bin1)
#                         hypercholesterolema_bins.append(bin2)
#                     elif item == '250.00':
#                         diabetes_bins.append(bin1)
#                         diabetes_bins.append(bin2)

            # search for significant similarity in Jscore (significance arb set as 0.1 for now)
#             people_compared.setdefault((rando1,rando2),[])

            # for each pair of people compared, we are putting in the score for each of their bin comparisons
#             people_compared[(rando1, rando2)].append({(bin1, bin2) : jsim})

            bin_scores.append(jsim)
            bin_comparisons.append((bin1, bin2))
            jscore_to_bin_dict[jsim] = (bin1, bin2)
        similarity_scores_matrix.append(bin_scores)
        bins_compared.append(bin_comparisons)


    #row indexes correspond to indexes of person1_bin_set, column indexes correspond to person2_bin_set indexes

    #part 2:
    max_value = 0.0
    max_row = 0
    max_column = 0
    exclude_rows = []
    exclude_columns = []
    max_links = []
    maxes = []
    excluded_bins = []
    max_dict_ordered = OrderedDict()
    excluded_bins_to_raw_score = OrderedDict()


    similarity_scores_matrix_array = np.array(similarity_scores_matrix)

    similarity_matrix = np.asmatrix(similarity_scores_matrix_array)

    # goal is to make sure that the two bins compared are associated with the best score

    bins_compared = np.array(bins_compared)

    while len(similarity_scores_matrix_array)>0:

        max_index = np.argmax(similarity_scores_matrix_array)

        max_score = similarity_matrix.max()

        max_row, max_col = np.unravel_index(max_index, similarity_scores_matrix_array.shape)

        exclude_rows.append(max_row)
        exclude_columns.append(max_col)
        maxes.append(max_score)

        tupleKey = (bins_compared[max_row][max_col][0],bins_compared[max_row][max_col][1])

        excluded_bins_to_raw_score.setdefault(tupleKey,[])
        excluded_bins_to_raw_score[tupleKey].append(similarity_scores_matrix_array[max_row][max_col])

        max_dict_ordered[(max_row, max_col)] = max_score

        similarity_scores_matrix_array = np.delete(similarity_scores_matrix_array,max_row,0)
        similarity_scores_matrix_array = np.delete(similarity_scores_matrix_array,max_col,1)

        bins_compared = np.delete(bins_compared,max_row, 0)
        bins_compared = np.delete(bins_compared,max_col, 1)

        similarity_matrix = np.delete(similarity_matrix,max_row, 0)
        similarity_matrix = np.delete(similarity_matrix, max_col, 1)

        if similarity_scores_matrix_array.size == 0:
            break

    #part 4
    # Sort the keys
    sorted_keys = sorted(max_dict_ordered.keys(), reverse=True)

    # Create a new ordered dictionary with sorted keys
    sorted_ordered_dict = OrderedDict((key, max_dict_ordered[key]) for key in sorted_keys)
    links_to_graph = []
    for key1, value in sorted_ordered_dict.items():
        links_to_graph.append(value)

    averages_to_actually_graph = []
    # introduce an expand
    # eliminate
    # x is the bin we took out, and y is the average score

    while sorted_ordered_dict:
        avg_2_plot = [sorted_ordered_dict[key] for key in sorted_ordered_dict]
        averages_to_actually_graph.append(st.mean(avg_2_plot))
        indexes2use = [key for key in sorted_ordered_dict]
        indexes2actuallyplot.append(indexes2use)
        sorted_ordered_dict.popitem()

    sorted_keys = sorted(excluded_bins_to_raw_score.keys(), reverse=True)
    sorted_bins_to_max_score_dict = OrderedDict((key, excluded_bins_to_raw_score[key]) for key in sorted_keys)
    while sorted_bins_to_max_score_dict:
        maxes_from_matrix = [item for key in sorted_bins_to_max_score_dict for item in sorted_bins_to_max_score_dict[key]]
        bins_from_matrix = [key for key in sorted_bins_to_max_score_dict]
        maxes_average = st.mean(maxes_from_matrix)
        if (rando1, rando2) not in averages_with_bins:
            averages_with_bins[(rando1, rando2)] = []
        averages_with_bins[(rando1, rando2)].append({bins_from_matrix[-1] : maxes_average})
        sorted_bins_to_max_score_dict.popitem()
    person1_best_scores = {}
    person2_best_scores = {}
    #splitting the dictionary
    for key, value in excluded_bins_to_raw_score.items():
        person1_best_scores[key[0]] = value[0]
        person2_best_scores[key[1]] = value[0]
    sorted_by_bins_person1 = dict(sorted(person1_best_scores.items()))
    sorted_by_bins_person2 = dict(sorted(person2_best_scores.items()))
    if (rando1, rando2) not in split_avg:
        split_avg[(rando1, rando2)] = []
    curr_average = 0
    while sorted_by_bins_person2:
        closest_to_diagnosis_p2 = min(sorted_by_bins_person2)
        values = sorted_by_bins_person2.values()
        total = sum(values)
        curr_average = total / len(values)
        split_avg[(rando1, rando2)].append({closest_to_diagnosis_p2:curr_average})
        sorted_by_bins_person2.pop(closest_to_diagnosis_p2)

    new_averages = []
    y_intercept = averages_to_actually_graph[0]
    for item in averages_to_actually_graph:
        new_averages.append(item - y_intercept)
    multiple_links.append(new_averages)
print(split_avg)

In [ ]:
#DON'T RUN NOT SET UP
#figure this out to save histogram to a file thing

# This snippet assumes you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe = df

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename = 'test.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# save dataframe in a csv file in the same workspace as the notebook
my_dataframe.to_csv(destination_filename, index=False)

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file to the bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/data/"]
output = subprocess.run(args, capture_output=True)

# print output from gsutil
output.stderr

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Number of individuals
num_individuals = 100

# Sample unique individuals
unique_people = binned_individuals['person_id'].sample(num_individuals, replace=False).tolist()

# Precompute bin-concept mappings for each person
person_bin_map = {
    person: {
        bin_: frozenset(group['concept_code'])  # Convert to frozenset for fast set operations
        for bin_, group in df.groupby('bin')
    }
    for person, df in binned_individuals.groupby('person_id') if person in unique_people
}

# Initialize similarity matrix
similarity_matrix_100x100 = np.eye(num_individuals)  # Identity matrix, 1s on diagonal

# Compare each pair only once (upper triangle)
for i in range(num_individuals):
    person1 = unique_people[i]
    bins1 = person_bin_map[person1]

    for j in range(i + 1, num_individuals):
        person2 = unique_people[j]
        bins2 = person_bin_map[person2]

        # Compute max Jaccard similarity for all bin pairs
        max_similarity_score = max(
            len(bins1[b1] & bins2[b2]) / len(bins1[b1] | bins2[b2]) if len(bins1[b1] | bins2[b2]) > 0 else 0
            for b1 in bins1 for b2 in bins2
        ) if bins1 and bins2 else 0

        # Fill symmetric similarity matrix
        similarity_matrix_100x100[i, j] = max_similarity_score
        similarity_matrix_100x100[j, i] = max_similarity_score  # Mirror value

# ** Generate Heatmap **
plt.figure(figsize=(12, 8))

# Assign heatmap to a variable
# heatmap = sns.heatmap(similarity_matrix_100x100, cmap="coolwarm",
#                       annot=False, xticklabels=False, yticklabels=False)

sns.clustermap(
    similarity_matrix_100x100,
    cmap="coolwarm",
    row_cluster=True,  # Cluster rows
    col_cluster=True,  # Cluster columns
    annot=False,
    xticklabels=False,
    yticklabels=False,
    figsize=(12, 8),
    cbar_kws={'label': "Jaccard Similarity Score"}
)

plt.title("Pairwise Similarity Heatmap (100x100)")
plt.xlabel("Individual Index")
plt.ylabel("Individual Index")

plt.show()

In [ ]:
# Working on Optimization
from collections import OrderedDict
import statistics as st
import numpy as np
import itertools
import copy

indexes2actuallyplot = []
jscore_to_bin_dict = {}
matching_codes = []
people_compared = {}
people_bins_avgScores = {}
averages_with_bins = {}
split_avg = {}

for _ in range(100):
    rando1 = binned_individuals['person_id'].sample().iloc[0]
    rando2 = binned_individuals['person_id'].sample().iloc[0]
    person1_df = binned_individuals[binned_individuals['person_id'] == rando1]
    bin_values_person1 = person1_df['bin'].tolist()

    person1_bin_set = set(bin_values_person1)
    person2_df = binned_individuals[binned_individuals['person_id'] == rando2]
    bin_values_person2 = person2_df['bin'].tolist()

    person2_bin_set = set(bin_values_person2)

    similarity_scores_matrix = []
    bins_compared = []

    for bin1 in person1_bin_set:
        bin_scores = []
        bin_comparisons = []
        for bin2 in person2_bin_set:
            Asub = set(person1_df.loc[(person1_df['bin'] == bin1), 'concept_code'])
            Bsub = set(person2_df.loc[(person2_df['bin'] == bin2), 'concept_code'])
            C = Asub.intersection(Bsub)
            D = Asub.union(Bsub)
            if len(D) == 0:
                jsim = 0.0
            else:
                jsim = float(len(C))/float(len(D))
            if len(C) > 0:
                matching_codes.append(C)
            bin_scores.append(jsim)
            bin_comparisons.append((bin1, bin2))
            jscore_to_bin_dict[jsim] = (bin1, bin2)
        similarity_scores_matrix.append(bin_scores)
        bins_compared.append(bin_comparisons)

    max_value = 0.0
    max_row = 0
    max_column = 0
    exclude_rows = []
    exclude_columns = []
    max_links = []
    maxes = []
    excluded_bins = []
    max_dict_ordered = OrderedDict()
    excluded_bins_to_raw_score = OrderedDict()

    similarity_scores_matrix_array = np.array(similarity_scores_matrix)
    similarity_matrix = np.asmatrix(similarity_scores_matrix_array)

    bins_compared = np.array(bins_compared)

    while len(similarity_scores_matrix_array)>0:
        max_index = np.argmax(similarity_scores_matrix_array)
        max_score = similarity_matrix.max()
        max_row, max_col = np.unravel_index(max_index, similarity_scores_matrix_array.shape)

        exclude_rows.append(max_row)
        exclude_columns.append(max_col)
        maxes.append(max_score)

        tupleKey = (bins_compared[max_row][max_col][0],bins_compared[max_row][max_col][1])

        excluded_bins_to_raw_score.setdefault(tupleKey,[])
        excluded_bins_to_raw_score[tupleKey].append(similarity_scores_matrix_array[max_row][max_col])

        max_dict_ordered[(max_row, max_col)] = max_score

        similarity_scores_matrix_array = np.delete(similarity_scores_matrix_array,max_row,0)
        similarity_scores_matrix_array = np.delete(similarity_scores_matrix_array,max_col,1)

        bins_compared = np.delete(bins_compared,max_row, 0)
        bins_compared = np.delete(bins_compared,max_col, 1)

        similarity_matrix = np.delete(similarity_matrix,max_row, 0)
        similarity_matrix = np.delete(similarity_matrix, max_col, 1)
        if similarity_scores_matrix_array.size == 0:
            break

    sorted_keys = sorted(max_dict_ordered.keys(), reverse=True)
    sorted_ordered_dict = OrderedDict((key, max_dict_ordered[key]) for key in sorted_keys)
    links_to_graph = []
    for key1, value in sorted_ordered_dict.items():
        links_to_graph.append(value)

    averages_to_actually_graph = []
    while sorted_ordered_dict:
        avg_2_plot = [sorted_ordered_dict[key] for key in sorted_ordered_dict]
        averages_to_actually_graph.append(st.mean(avg_2_plot))
        indexes2use = [key for key in sorted_ordered_dict]
        indexes2actuallyplot.append(indexes2use)
        sorted_ordered_dict.popitem()

    sorted_keys = sorted(excluded_bins_to_raw_score.keys(), reverse=True)
    sorted_bins_to_max_score_dict = OrderedDict((key, excluded_bins_to_raw_score[key]) for key in sorted_keys)
    while sorted_bins_to_max_score_dict:
        maxes_from_matrix = [item for key in sorted_bins_to_max_score_dict for item in sorted_bins_to_max_score_dict[key]]
        bins_from_matrix = [key for key in sorted_bins_to_max_score_dict]
        maxes_average = st.mean(maxes_from_matrix)
        if (rando1, rando2) not in averages_with_bins:
            averages_with_bins[(rando1, rando2)] = []
        averages_with_bins[(rando1, rando2)].append({bins_from_matrix[-1] : maxes_average})
        sorted_bins_to_max_score_dict.popitem()
    person1_best_scores = {}
    person2_best_scores = {}

    for key, value in excluded_bins_to_raw_score.items():
        person1_best_scores[key[0]] = value[0]
        person2_best_scores[key[1]] = value[0]
    sorted_by_bins_person1 = dict(sorted(person1_best_scores.items()))
    sorted_by_bins_person2 = dict(sorted(person2_best_scores.items()))
    if (rando1, rando2) not in split_avg:
        split_avg[(rando1, rando2)] = []
    curr_average = 0
    while sorted_by_bins_person2:
        closest_to_diagnosis_p2 = min(sorted_by_bins_person2)
        values = sorted_by_bins_person2.values()
        total = sum(values)
        curr_average = total / len(values)
        split_avg[(rando1, rando2)].append({closest_to_diagnosis_p2:curr_average})
        sorted_by_bins_person2.pop(closest_to_diagnosis_p2)

print(split_avg)

In [ ]:
#2000 in 4 1/2 minutes Optimized
from collections import OrderedDict
import statistics as st
import numpy as np

indexes2actuallyplot = []
jscore_to_bin_dict = {}
matching_codes = []
people_compared = {}
people_bins_avgScores = {}
averages_with_bins = {}
split_avg = {}

for _ in range(2000):
    rando1 = binned_individuals['person_id'].sample().iloc[0]
    rando2 = binned_individuals['person_id'].sample().iloc[0]
    person1_df = binned_individuals[binned_individuals['person_id'] == rando1]
    person2_df = binned_individuals[binned_individuals['person_id'] == rando2]

    person1_bin_set = set(person1_df['bin'])
    person2_bin_set = set(person2_df['bin'])

    bin_pairs = np.array([(bin1, bin2) for bin1 in person1_bin_set for bin2 in person2_bin_set], dtype=object)

    Asub_list = {bin1: set(person1_df.loc[person1_df['bin'] == bin1, 'concept_code']) for bin1 in person1_bin_set}
    Bsub_list = {bin2: set(person2_df.loc[person2_df['bin'] == bin2, 'concept_code']) for bin2 in person2_bin_set}

    similarity_scores = np.array([
        len(Asub_list[bin1] & Bsub_list[bin2]) / len(Asub_list[bin1] | Bsub_list[bin2]) if len(Asub_list[bin1] | Bsub_list[bin2]) > 0 else 0.0
        for bin1, bin2 in bin_pairs
    ])

    similarity_matrix = similarity_scores.reshape(len(person1_bin_set), len(person2_bin_set))
    mask = np.ones(similarity_matrix.shape, dtype=bool)
    max_dict_ordered = OrderedDict()
    excluded_bins_to_raw_score = OrderedDict()

    while np.any(mask):
        masked_matrix = np.where(mask, similarity_matrix, -np.inf)
        max_index = np.argmax(masked_matrix)
        max_row, max_col = np.unravel_index(max_index, similarity_matrix.shape)
        max_score = similarity_matrix[max_row, max_col]

        tuple_key = tuple(bin_pairs[max_row * len(person2_bin_set) + max_col])
        max_dict_ordered[tuple_key] = max_score
        excluded_bins_to_raw_score.setdefault(tuple_key, []).append(max_score)

        mask[max_row, :] = False
        mask[:, max_col] = False

    sorted_ordered_dict = OrderedDict(sorted(max_dict_ordered.items(), key=lambda x: x[1], reverse=True))
    indexes2actuallyplot.append(list(sorted_ordered_dict.keys()))
    averages_to_actually_graph = [st.mean(sorted_ordered_dict.values())]

    sorted_bins_to_max_score_dict = OrderedDict(sorted(excluded_bins_to_raw_score.items(), key=lambda x: max(x[1]), reverse=True))
    while sorted_bins_to_max_score_dict:
        bins_from_matrix = list(sorted_bins_to_max_score_dict.keys())
        maxes_average = st.mean([val for values in sorted_bins_to_max_score_dict.values() for val in values])
        averages_with_bins.setdefault((rando1, rando2), []).append({bins_from_matrix[-1]: maxes_average})
        sorted_bins_to_max_score_dict.popitem()

    person1_best_scores = {key[0]: value[0] for key, value in excluded_bins_to_raw_score.items()}
    person2_best_scores = {key[1]: value[0] for key, value in excluded_bins_to_raw_score.items()}

    sorted_by_bins_person1 = OrderedDict(sorted(person1_best_scores.items()))
    sorted_by_bins_person2 = OrderedDict(sorted(person2_best_scores.items()))

    if (rando1, rando2) not in split_avg:
        split_avg[(rando1, rando2)] = []

    while sorted_by_bins_person2:
        closest_to_diagnosis_p2 = next(iter(sorted_by_bins_person2))
        curr_average = st.mean(sorted_by_bins_person2.values())
        split_avg[(rando1, rando2)].append({closest_to_diagnosis_p2: curr_average})
        del sorted_by_bins_person2[closest_to_diagnosis_p2]

print(split_avg)

In [ ]:
control_dataframe.columns

# THE BELOW CODE IS FOR COEFFICIENT ANALYSIS FOR CONTROL GROUP (OBSOLETE)

Results of this code are stored in 'control_group_coordinate_peaks.csv'

In [ ]:
#pull in control group people

import os
import subprocess
import numpy as np
import pandas as pd

# This snippet assumes you run setup first

# This code copies file in your Google Bucket and loads it into a dataframe

# Replace 'test.csv' with THE NAME of the file you're going to download from the bucket (don't delete the quotation marks)
name_of_file_in_bucket = 'control_group_malignant.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
control_dataframe = pd.read_csv(name_of_file_in_bucket)
control_dataframe.head()

print(len(control_dataframe))

In [ ]:
# Bins each event
control_dataframe['bin'] = pd.cut(control_dataframe['day_diff'],labels=labels,bins=bins)
control_dataframe.head()

value_counts = control_dataframe['bin'].value_counts().sort_index().reset_index()
value_counts.head(10)
num_events = value_counts['count'].tolist()
print(num_events)

control_dataframe = control_dataframe[pd.notna(control_dataframe['bin'])]
control_dataframe['concept_code'] = control_dataframe['standard_concept_code']
control_dataframe.head(30)

Optimized "4.5 minute" Jaccard Scorer

In [ ]:
#2000 in 4 1/2 minutes Optimized
from collections import OrderedDict
import statistics as st
import numpy as np

indexes2actuallyplot = []
jscore_to_bin_dict = {}
matching_codes = []
people_compared = {}
people_bins_avgScores = {}
averages_with_bins = {}
split_avg = {}

for _ in range(100):
    rando1 = control_dataframe['person_id'].sample().iloc[0]
    rando2 = control_dataframe['person_id'].sample().iloc[0]
    person1_df = control_dataframe[control_dataframe['person_id'] == rando1]
    person2_df = control_dataframe[control_dataframe['person_id'] == rando2]

    person1_bin_set = set(person1_df['bin'])
    person2_bin_set = set(person2_df['bin'])

    bin_pairs = np.array([(bin1, bin2) for bin1 in person1_bin_set for bin2 in person2_bin_set], dtype=object)

    Asub_list = {bin1: set(person1_df.loc[person1_df['bin'] == bin1, 'concept_code']) for bin1 in person1_bin_set}
    Bsub_list = {bin2: set(person2_df.loc[person2_df['bin'] == bin2, 'concept_code']) for bin2 in person2_bin_set}

    similarity_scores = np.array([
        len(Asub_list[bin1] & Bsub_list[bin2]) / len(Asub_list[bin1] | Bsub_list[bin2]) if len(Asub_list[bin1] | Bsub_list[bin2]) > 0 else 0.0
        for bin1, bin2 in bin_pairs
    ])

    similarity_matrix = similarity_scores.reshape(len(person1_bin_set), len(person2_bin_set))
    mask = np.ones(similarity_matrix.shape, dtype=bool)
    max_dict_ordered = OrderedDict()
    excluded_bins_to_raw_score = OrderedDict()

    while np.any(mask):
        masked_matrix = np.where(mask, similarity_matrix, -np.inf)
        max_index = np.argmax(masked_matrix)
        max_row, max_col = np.unravel_index(max_index, similarity_matrix.shape)
        max_score = similarity_matrix[max_row, max_col]

        tuple_key = tuple(bin_pairs[max_row * len(person2_bin_set) + max_col])
        max_dict_ordered[tuple_key] = max_score
        excluded_bins_to_raw_score.setdefault(tuple_key, []).append(max_score)

        mask[max_row, :] = False
        mask[:, max_col] = False

    sorted_ordered_dict = OrderedDict(sorted(max_dict_ordered.items(), key=lambda x: x[1], reverse=True))
    indexes2actuallyplot.append(list(sorted_ordered_dict.keys()))
    averages_to_actually_graph = [st.mean(sorted_ordered_dict.values())]

    sorted_bins_to_max_score_dict = OrderedDict(sorted(excluded_bins_to_raw_score.items(), key=lambda x: max(x[1]), reverse=True))
    while sorted_bins_to_max_score_dict:
        bins_from_matrix = list(sorted_bins_to_max_score_dict.keys())
        maxes_average = st.mean([val for values in sorted_bins_to_max_score_dict.values() for val in values])
        averages_with_bins.setdefault((rando1, rando2), []).append({bins_from_matrix[-1]: maxes_average})
        sorted_bins_to_max_score_dict.popitem()

    person1_best_scores = {key[0]: value[0] for key, value in excluded_bins_to_raw_score.items()}
    person2_best_scores = {key[1]: value[0] for key, value in excluded_bins_to_raw_score.items()}

    sorted_by_bins_person1 = OrderedDict(sorted(person1_best_scores.items()))
    sorted_by_bins_person2 = OrderedDict(sorted(person2_best_scores.items()))

    if (rando1, rando2) not in split_avg:
        split_avg[(rando1, rando2)] = []

    while sorted_by_bins_person2:
        closest_to_diagnosis_p2 = next(iter(sorted_by_bins_person2))
        curr_average = st.mean(sorted_by_bins_person2.values())
        split_avg[(rando1, rando2)].append({closest_to_diagnosis_p2: curr_average})
        del sorted_by_bins_person2[closest_to_diagnosis_p2]

print(split_avg)

In [ ]:
result = []
comparison_ids = [] # This is storing tuples
# the key is the person, the person is the dictionary of bins to scores
for key, value in split_avg.items():
    comparison_ids.append(key)
    person2_bins = {}
    for val in value:
        for k, v in val.items():
            bin = int(k.split(',')[0])
            person2_bins[bin] = v
    scores = {}
    score_list = []
    curr_score = 0
    for j in bins:
            curr_bin = j
            if j not in person2_bins:
                curr_score = curr_score
            else:
                curr_score = person2_bins[j]
            scores[curr_bin] = curr_score
            score_list.append(curr_score)
    max_score = 0
    for item in score_list:
        if item > max_score:
            max_score = item
    if max_score != 1.0:
        result.append(scores)

len(result)

Extracting Coefficients and Verteces

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt


comparison_functions = {}

i = -1
for dataset in result:
    i += 1
    x = np.array(list(dataset.keys()))
    y = np.array(list(dataset.values()))

    # This sucker is for finding the last nonzero value
    index = 0
    while index<len(y) and y[index]==0:
        index+=1
    y_cutoff = y[index:]
    x_cutoff = x[index:]
    if x_cutoff.size == 0:
        continue
    coefficients = np.polyfit(x_cutoff,y_cutoff,2)
    comparison_functions[comparison_ids[i]] = coefficients

len(comparison_functions)

In [ ]:
# takes in the 3 quadratic coefficients and returns the max/min based on the derivative
def get_quadratic_peak(A,B,C):
    peak_x = (-1*B)/(2*A)
    peak_y = A*peak_x**2+ B*peak_x + C
    return peak_x, peak_y

coordinate_peaks = {}
for person_id, coefficients in comparison_functions.items():
    A,B,C = coefficients
    if A == 0:
        continue
    peak = get_quadratic_peak(A,B,C)
    coordinate_peaks[person_id] = peak

In [ ]:
control_peak_coordinates = pd.DataFrame({"PeopleCompared": list(coordinate_peaks.keys()), "Peak_Coordinate":list(coordinate_peaks.values())})

# Split the tuple into two columns
control_peak_coordinates[['X_Peak_Coordinate', 'Y_Peak_Coordinate']] = pd.DataFrame(control_peak_coordinates['Peak_Coordinate'].tolist(), index=similarity_peak_coordinates.index)

# Drop the original column if not needed
control_peak_coordinates.drop(columns=['Peak_Coordinate'], inplace=True)

control_peak_coordinates

Saving to a destination file in the bucket

In [ ]:
# This snippet assumes you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe = control_peak_coordinates

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename = 'control_group_coordinate_peaks.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# save dataframe in a csv file in the same workspace as the notebook
my_dataframe.to_csv(destination_filename, index=False)

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file to the bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/data/"]
output = subprocess.run(args, capture_output=True)

# print output from gsutil
output.stderr

Below is the Heatmap Code

In [ ]:
#control group comparisons
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Get all unique individuals in the dataframe
unique_people = control_dataframe['person_id'].unique().tolist()
num_individuals = len(unique_people)

# Sample unique individuals
unique_people = control_dataframe['person_id'].sample(num_individuals, replace=False).tolist()

# Precompute bin-concept mappings for each person
person_bin_map = {
    person: {
        bin_: frozenset(group['standard_concept_code'])  # Convert to frozenset for fast set operations
        for bin_, group in df.groupby('bin')
    }
    for person, df in control_dataframe.groupby('person_id') if person in unique_people
}

# Initialize similarity matrix
similarity_matrix_100x100 = np.eye(num_individuals)  # Identity matrix, 1s on diagonal

# Compare each pair only once
for i in range(num_individuals):
    person1 = unique_people[i]
    bins1 = person_bin_map[person1]

    for j in range(i + 1, num_individuals):
        person2 = unique_people[j]
        bins2 = person_bin_map[person2]

        # Compute max Jaccard similarity for all bin pairs
        max_similarity_score = max(
            len(bins1[b1] & bins2[b2]) / len(bins1[b1] | bins2[b2]) if len(bins1[b1] | bins2[b2]) > 0 else 0
            for b1 in bins1 for b2 in bins2
        ) if bins1 and bins2 else 0

        # Fill symmetric similarity matrix
        similarity_matrix_100x100[i, j] = max_similarity_score
        similarity_matrix_100x100[j, i] = max_similarity_score  # Mirror value

# ** Generate Heatmap **
plt.figure(figsize=(12, 8))

# Assign heatmap to a variable
# heatmap = sns.heatmap(similarity_matrix_100x100, cmap="coolwarm",
#                       annot=False, xticklabels=False, yticklabels=False)

sns.clustermap(
    similarity_matrix_100x100,
    cmap="coolwarm",
    row_cluster=True,  # Cluster rows
    col_cluster=True,  # Cluster columns
    annot=False,
    xticklabels=False,
    yticklabels=False,
    figsize=(12, 8),
    cbar_kws={'label': "Jaccard Similarity Score"}
)

plt.title("Pairwise Similarity Heatmap (100x100)")
plt.xlabel("Individual Index")
plt.ylabel("Individual Index")

plt.show()

In [ ]:
counts_df = control_dataframe.groupby('person_id').size().reset_index(name='count')
print(np.mean(counts_df['count']))
print(np.median(counts_df['count']))
filtered_counts = counts_df[counts_df['count'] > 100]
ids = filtered_counts['person_id']
filtered_control = control_dataframe[control_dataframe['person_id'].isin(ids)]
print(len(filtered_control["person_id"].unique().tolist()))

In [ ]:
#control group comparisons
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Get all unique individuals in the dataframe
unique_people = filtered_control['person_id'].unique().tolist()
num_individuals = len(unique_people)

# Sample unique individuals
unique_people = filtered_control['person_id'].sample(num_individuals, replace=False).tolist()

# Precompute bin-concept mappings for each person
person_bin_map = {
    person: {
        bin_: frozenset(group['standard_concept_code'])  # Convert to frozenset for fast set operations
        for bin_, group in df.groupby('bin')
    }
    for person, df in filtered_control.groupby('person_id') if person in unique_people
}

# Initialize similarity matrix
similarity_matrix_100x100 = np.eye(num_individuals)  # Identity matrix, 1s on diagonal

# Compare each pair only once
for i in range(num_individuals):
    person1 = unique_people[i]
    bins1 = person_bin_map[person1]

    for j in range(i + 1, num_individuals):
        person2 = unique_people[j]
        bins2 = person_bin_map[person2]

        # Compute average Jaccard similarity for all bin pairs
        similarity_scores = [
            len(bins1[b1] & bins2[b2]) / len(bins1[b1] | bins2[b2]) if len(bins1[b1] | bins2[b2]) > 0 else 0
            for b1 in bins1 for b2 in bins2
        ]

        # Take the average of all scores, or 0 if there are no valid comparisons
        avg_similarity_score = sum(similarity_scores) if similarity_scores else 0

        # Fill symmetric similarity matrix
        similarity_matrix_100x100[i, j] = avg_similarity_score
        similarity_matrix_100x100[j, i] = avg_similarity_score  # Mirror value

# ** Generate Heatmap **
plt.figure(figsize=(12, 8))

sns.clustermap(
    similarity_matrix_100x100,
    cmap="coolwarm",
    row_cluster=True,  # Cluster rows
    col_cluster=True,  # Cluster columns
    annot=False,
    xticklabels=False,
    yticklabels=False,
    figsize=(12, 8),
    cbar_kws={'label': "Jaccard Similarity Score"},

)

plt.title("Control Group -- Sum Jaccard Scores")
plt.xlabel("Individual Index")
plt.ylabel("Individual Index")

plt.show()

# COEFFICIENT ANALYSIS FOR 83 CONTROLLED PEOPLE

Results of this code are stored in '2000_randos_peaks.csv'

In [ ]:
#control actual average comparisons
from collections import OrderedDict
import statistics as st
import numpy as np
from itertools import combinations

fixed_people = filtered_control['person_id'].drop_duplicates().tolist()

indexes2actuallyplot = []
jscore_to_bin_dict = {}
matching_codes = []
people_compared = {}
people_bins_avgScores = {}
averages_with_bins = {}
split_avg = {}

for rando1, rando2 in combinations(fixed_people, 2):
    person1_df = filtered_control[filtered_control['person_id'] == rando1]
    person2_df = filtered_control[filtered_control['person_id'] == rando2]

    person1_bin_set = set(person1_df['bin'])
    person2_bin_set = set(person2_df['bin'])

    bin_pairs = np.array([(bin1, bin2) for bin1 in person1_bin_set for bin2 in person2_bin_set], dtype=object)

    Asub_list = {bin1: set(person1_df.loc[person1_df['bin'] == bin1, 'standard_concept_code']) for bin1 in person1_bin_set}
    Bsub_list = {bin2: set(person2_df.loc[person2_df['bin'] == bin2, 'standard_concept_code']) for bin2 in person2_bin_set}

    similarity_scores = np.array([
        len(Asub_list[bin1] & Bsub_list[bin2]) / len(Asub_list[bin1] | Bsub_list[bin2]) if len(Asub_list[bin1] | Bsub_list[bin2]) > 0 else 0.0
        for bin1, bin2 in bin_pairs
    ])

    similarity_matrix = similarity_scores.reshape(len(person1_bin_set), len(person2_bin_set))
    mask = np.ones(similarity_matrix.shape, dtype=bool)
    max_dict_ordered = OrderedDict()
    excluded_bins_to_raw_score = OrderedDict()

    while np.any(mask):
        masked_matrix = np.where(mask, similarity_matrix, -np.inf)
        max_index = np.argmax(masked_matrix)
        max_row, max_col = np.unravel_index(max_index, similarity_matrix.shape)
        max_score = similarity_matrix[max_row, max_col]

        tuple_key = tuple(bin_pairs[max_row * len(person2_bin_set) + max_col])
        max_dict_ordered[tuple_key] = max_score
        excluded_bins_to_raw_score.setdefault(tuple_key, []).append(max_score)

        mask[max_row, :] = False
        mask[:, max_col] = False

    sorted_ordered_dict = OrderedDict(sorted(max_dict_ordered.items(), key=lambda x: x[1], reverse=True))
    indexes2actuallyplot.append(list(sorted_ordered_dict.keys()))
    averages_to_actually_graph = [st.mean(sorted_ordered_dict.values())]

    sorted_bins_to_max_score_dict = OrderedDict(sorted(excluded_bins_to_raw_score.items(), key=lambda x: max(x[1]), reverse=True))
    while sorted_bins_to_max_score_dict:
        bins_from_matrix = list(sorted_bins_to_max_score_dict.keys())
        maxes_average = st.mean([val for values in sorted_bins_to_max_score_dict.values() for val in values])
        averages_with_bins.setdefault((rando1, rando2), []).append({bins_from_matrix[-1]: maxes_average})
        sorted_bins_to_max_score_dict.popitem()

    person1_best_scores = {key[0]: value[0] for key, value in excluded_bins_to_raw_score.items()}
    person2_best_scores = {key[1]: value[0] for key, value in excluded_bins_to_raw_score.items()}

    sorted_by_bins_person1 = OrderedDict(sorted(person1_best_scores.items()))
    sorted_by_bins_person2 = OrderedDict(sorted(person2_best_scores.items()))

    if (rando1, rando2) not in split_avg:
        split_avg[(rando1, rando2)] = []

    while sorted_by_bins_person2:
        closest_to_diagnosis_p2 = next(iter(sorted_by_bins_person2))
        curr_average = st.mean(sorted_by_bins_person2.values())
        split_avg[(rando1, rando2)].append({closest_to_diagnosis_p2: curr_average})
        del sorted_by_bins_person2[closest_to_diagnosis_p2]

print(split_avg)

In [ ]:
result = []
comparison_ids = [] # This is storing tuples
# the key is the person, the person is the dictionary of bins to scores
for key, value in split_avg.items():
    comparison_ids.append(key)
    person2_bins = {}
    for val in value:
        for k, v in val.items():
            bin = int(k.split(',')[0])
            person2_bins[bin] = v
    scores = {}
    score_list = []
    curr_score = 0
    for j in bins:
            curr_bin = j
            if j not in person2_bins:
                curr_score = curr_score
            else:
                curr_score = person2_bins[j]
            scores[curr_bin] = curr_score
            score_list.append(curr_score)
    max_score = 0
    for item in score_list:
        if item > max_score:
            max_score = item
    if max_score != 1.0:
        result.append(scores)

The function below this is going to be ONLY for grabbing coefficients

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt


comparison_functions = {}

i = -1
for dataset in result:
    i += 1
    x = np.array(list(dataset.keys()))
    y = np.array(list(dataset.values()))

    # This sucker is for finding the last nonzero value
    index = 0
    while index<len(y) and y[index]==0:
        index+=1


    y_cutoff = y[index:]
    x_cutoff = x[index:]
    if x_cutoff.size == 0:
        continue
    coefficients = np.polyfit(x_cutoff,y_cutoff,2)
    comparison_functions[comparison_ids[i]] = coefficients

In [ ]:
# takes in the 3 quadratic coefficients and returns the max/min based on the derivative
def get_quadratic_peak(A,B,C):
    peak_x = (-1*B)/(2*A)
    peak_y = A*peak_x**2+ B*peak_x + C
    return peak_x, peak_y

coordinate_peaks = {}
for person_id, coefficients in comparison_functions.items():
    A,B,C = coefficients
#     print(coefficients)
    if A == 0:
        continue
    peak = get_quadratic_peak(A,B,C)
    coordinate_peaks[person_id] = peak

In [ ]:
similarity_peak_coordinates = pd.DataFrame({"PeopleCompared": list(coordinate_peaks.keys()), "Peak_Coordinate":list(coordinate_peaks.values())})

# Split the tuple into two columns
similarity_peak_coordinates[['X_Peak_Coordinate', 'Y_Peak_Coordinate']] = pd.DataFrame(similarity_peak_coordinates['Peak_Coordinate'].tolist(), index=similarity_peak_coordinates.index)

# Drop the original column if not needed
similarity_peak_coordinates.drop(columns=['Peak_Coordinate'], inplace=True)

similarity_peak_coordinates.head()

BELOW: Saving our similarity peaks for the 2000 random people!

Name: '2000_randos_peaks.csv'

In [ ]:
# This snippet assumes you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe = similarity_peak_coordinates

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename = '2000_randos_peaks.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# save dataframe in a csv file in the same workspace as the notebook
my_dataframe.to_csv(destination_filename, index=False)

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file to the bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/data/"]
output = subprocess.run(args, capture_output=True)

# print output from gsutil
output.stderr

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

plt.figure(figsize=(20,14))

# coefficients = []
comparison_functions = {}

i = -1
for dataset in result:
    i += 1
    x = np.array(list(dataset.keys()))
    y = np.array(list(dataset.values()))

    # This sucker is for finding the last nonzero value
    index = 0
    while index<len(y) and y[index]==0:
        index+=1


    y_cutoff = y[index:]
    x_cutoff = x[index:]
    if x_cutoff.size == 0:
#         comparison_functions[comparison_ids[i]] = None
        continue
#     print(x_cutoff)
    coefficients = np.polyfit(x_cutoff,y_cutoff,2)
    comparison_functions[comparison_ids[i]] = coefficients
#     print(coefficients)

    quadratic_model = np.poly1d(coefficients)
    # maybe not x cutoff?
    y_pred = quadratic_model(x_cutoff)

    x_fit = np.linspace(x.min(), x.max(), 100)
    y_fit = quadratic_model(x_fit)
#     if tuple(coefficients)[0] > 8:

# RIGHT HERE: ONLY PLOTTING TOP QUARTILE SLOPES
    if coefficients[0] > 5.5052460536468e-08:
#         possible_misdiagnosis[key] = value
        plt.plot(x_fit, y_fit)
plt.gca().invert_xaxis()
plt.xlabel('Days Before Cancer Diagnosis (Bins)', fontsize = 20)
plt.ylabel('Jaccard Score', fontsize = 20)
plt.title('Control Group: 83 Participants, EHRs > 100 - Pairwise Comparisons', fontsize = 25)
plt.xlim(0,-3000)
plt.ylim(-.05,.25)
plt.grid(True)

# plt.savefig("PairwiseComparisons_v2.pdf", format="pdf")
#plt.savefig("PairwiseComparisons.png")



# plt.figure(figsize=(20,8))

# for dataset in result:
#     x = np.array(list(dataset.keys()))
#     y = np.array(list(dataset.values()))
#     coefficients = np.polyfit(x,y,2)
#     print(coefficients)


# #     print(y)
#     quadratic_model = np.poly1d(coefficients)
#     y_pred = quadratic_model(x)

#     x_fit = np.linspace(x.min(), x.max(), 100)
#     y_fit = quadratic_model(x_fit)
# #     print(y_fit)
#     mx = np.argmax(y_fit)
#     if coefficients[0] > 0:
#         mx = np.argmin(y_fit)

#     print(f"Quadratic Model local Min/Max = {x[mx]}")
#     print()
#     plt.plot(x_fit, y_fit)


# plt.gca().invert_xaxis()
# # Add labels and title
# plt.xlabel('Bins -15000 -> 0')
# plt.ylabel('Jccard Score')
# plt.title('Person 2 Bins vs Score')

# # Display the graph
# plt.grid(True)
plt.show()

In [ ]:
counts_ind = binned_individuals.groupby('person_id').size().reset_index(name='count')
print(np.mean(counts_ind['count']))
print(np.median(counts_ind['count']))
filtered_counts = counts_ind[counts_ind['count'] > 100]
ids = filtered_counts['person_id']
filtered_binned = binned_individuals[binned_individuals['person_id'].isin(ids)]
print(len(filtered_binned["person_id"].unique().tolist()))

# COEFFICIENT ANALYSIS OF 83 INDIVIDUALS

In [ ]:
#rando 83 comparison
from collections import OrderedDict
import statistics as st
import numpy as np
from itertools import combinations

# Get all unique individuals in the dataframe
unique_people = filtered_binned['person_id'].unique().tolist()
num_individuals = 83
unique_people = filtered_binned['person_id'].sample(num_individuals, replace=False).tolist()

indexes2actuallyplot = []
jscore_to_bin_dict = {}
matching_codes = []
people_compared = {}
people_bins_avgScores = {}
averages_with_bins = {}
split_avg = {}

for rando1, rando2 in combinations(unique_people, 2):
    person1_df = filtered_binned[filtered_binned['person_id'] == rando1]
    person2_df = filtered_binned[filtered_binned['person_id'] == rando2]

    person1_bin_set = set(person1_df['bin'])
    person2_bin_set = set(person2_df['bin'])

    bin_pairs = np.array([(bin1, bin2) for bin1 in person1_bin_set for bin2 in person2_bin_set], dtype=object)

    Asub_list = {bin1: set(person1_df.loc[person1_df['bin'] == bin1, 'concept_code']) for bin1 in person1_bin_set}
    Bsub_list = {bin2: set(person2_df.loc[person2_df['bin'] == bin2, 'concept_code']) for bin2 in person2_bin_set}

    similarity_scores = np.array([
        len(Asub_list[bin1] & Bsub_list[bin2]) / len(Asub_list[bin1] | Bsub_list[bin2]) if len(Asub_list[bin1] | Bsub_list[bin2]) > 0 else 0.0
        for bin1, bin2 in bin_pairs
    ])

    similarity_matrix = similarity_scores.reshape(len(person1_bin_set), len(person2_bin_set))
    mask = np.ones(similarity_matrix.shape, dtype=bool)
    max_dict_ordered = OrderedDict()
    excluded_bins_to_raw_score = OrderedDict()

    while np.any(mask):
        masked_matrix = np.where(mask, similarity_matrix, -np.inf)
        max_index = np.argmax(masked_matrix)
        max_row, max_col = np.unravel_index(max_index, similarity_matrix.shape)
        max_score = similarity_matrix[max_row, max_col]

        tuple_key = tuple(bin_pairs[max_row * len(person2_bin_set) + max_col])
        max_dict_ordered[tuple_key] = max_score
        excluded_bins_to_raw_score.setdefault(tuple_key, []).append(max_score)

        mask[max_row, :] = False
        mask[:, max_col] = False

    sorted_ordered_dict = OrderedDict(sorted(max_dict_ordered.items(), key=lambda x: x[1], reverse=True))
    indexes2actuallyplot.append(list(sorted_ordered_dict.keys()))
    averages_to_actually_graph = [st.mean(sorted_ordered_dict.values())]

    sorted_bins_to_max_score_dict = OrderedDict(sorted(excluded_bins_to_raw_score.items(), key=lambda x: max(x[1]), reverse=True))
    while sorted_bins_to_max_score_dict:
        bins_from_matrix = list(sorted_bins_to_max_score_dict.keys())
        maxes_average = st.mean([val for values in sorted_bins_to_max_score_dict.values() for val in values])
        averages_with_bins.setdefault((rando1, rando2), []).append({bins_from_matrix[-1]: maxes_average})
        sorted_bins_to_max_score_dict.popitem()

    person1_best_scores = {key[0]: value[0] for key, value in excluded_bins_to_raw_score.items()}
    person2_best_scores = {key[1]: value[0] for key, value in excluded_bins_to_raw_score.items()}

    sorted_by_bins_person1 = OrderedDict(sorted(person1_best_scores.items()))
    sorted_by_bins_person2 = OrderedDict(sorted(person2_best_scores.items()))

    if (rando1, rando2) not in split_avg:
        split_avg[(rando1, rando2)] = []

    while sorted_by_bins_person2:
        closest_to_diagnosis_p2 = next(iter(sorted_by_bins_person2))
        curr_average = st.mean(sorted_by_bins_person2.values())
        split_avg[(rando1, rando2)].append({closest_to_diagnosis_p2: curr_average})
        del sorted_by_bins_person2[closest_to_diagnosis_p2]

In [ ]:
result = []
comparison_ids = [] # This is storing tuples
# the key is the person, the person is the dictionary of bins to scores
for key, value in split_avg.items():
    comparison_ids.append(key)
    person2_bins = {}
    for val in value:
        for k, v in val.items():
            bin = int(k.split(',')[0])
            person2_bins[bin] = v
    scores = {}
    score_list = []
    curr_score = 0
    for j in bins:
            curr_bin = j
            if j not in person2_bins:
                curr_score = curr_score
            else:
                curr_score = person2_bins[j]
            scores[curr_bin] = curr_score
            score_list.append(curr_score)
    max_score = 0
    for item in score_list:
        if item > max_score:
            max_score = item
    if max_score != 1.0:
        result.append(scores)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt


comparison_functions = {}

i = -1
for dataset in result:
    i += 1
    x = np.array(list(dataset.keys()))
    y = np.array(list(dataset.values()))

    # This sucker is for finding the last nonzero value
    index = 0
    while index<len(y) and y[index]==0:
        index+=1


    y_cutoff = y[index:]
    x_cutoff = x[index:]
    if x_cutoff.size == 0:
        continue
    coefficients = np.polyfit(x_cutoff,y_cutoff,2)
    comparison_functions[comparison_ids[i]] = coefficients

In [ ]:
comparison_functions

In [ ]:
# takes in the 3 quadratic coefficients and returns the max/min based on the derivative
def get_quadratic_peak(A,B,C):
    peak_x = (-1*B)/(2*A)
    peak_y = A*peak_x**2+ B*peak_x + C
    return peak_x, peak_y

coordinate_peaks = {}
for person_id, coefficients in comparison_functions.items():
    A,B,C = coefficients
#     print(coefficients)
    if A == 0:
        continue
    peak = get_quadratic_peak(A,B,C)
    coordinate_peaks[person_id] = peak

In [ ]:
similarity_peak_coordinates = pd.DataFrame({"PeopleCompared": list(coordinate_peaks.keys()), "Peak_Coordinate":list(coordinate_peaks.values())})

# Split the tuple into two columns
similarity_peak_coordinates[['X_Peak_Coordinate', 'Y_Peak_Coordinate']] = pd.DataFrame(similarity_peak_coordinates['Peak_Coordinate'].tolist(), index=similarity_peak_coordinates.index)

# Drop the original column if not needed
similarity_peak_coordinates.drop(columns=['Peak_Coordinate'], inplace=True)

similarity_peak_coordinates.head()

In [ ]:
# This snippet assumes you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe = similarity_peak_coordinates

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename = '83_individuals_peaks.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# save dataframe in a csv file in the same workspace as the notebook
my_dataframe.to_csv(destination_filename, index=False)

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file to the bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/data/"]
output = subprocess.run(args, capture_output=True)

# print output from gsutil
output.stderr

# Access all 3 vertex peak files from the Bucket

In [ ]:
# This snippet assumes you run setup first

# This code copies file in your Google Bucket and loads it into a dataframe

# Replace 'test.csv' with THE NAME of the file you're going to download from the bucket (don't delete the quotation marks)
name_of_file_in_bucket = 'control_group_coordinate_peaks.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
control_group_coordinate_peaks = pd.read_csv(name_of_file_in_bucket)
control_group_coordinate_peaks.head()

In [ ]:
# This snippet assumes you run setup first

# This code copies file in your Google Bucket and loads it into a dataframe

# Replace 'test.csv' with THE NAME of the file you're going to download from the bucket (don't delete the quotation marks)
name_of_file_in_bucket = '2000_randos_peaks.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
randos_peaks = pd.read_csv(name_of_file_in_bucket)
randos_peaks.head()

In [ ]:
# This snippet assumes you run setup first

# This code copies file in your Google Bucket and loads it into a dataframe

# Replace 'test.csv' with THE NAME of the file you're going to download from the bucket (don't delete the quotation marks)
name_of_file_in_bucket = '83_individuals_peaks.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
random_83_peaks = pd.read_csv(name_of_file_in_bucket)
random_83_peaks.head()

In [ ]:
jfig, ax = plt.subplots(figsize=(8, 6))

plt.hist(control_group_coordinate_peaks['X_Peak_Coordinate'],bins=bins,label="Control Group",alpha=0.3)
plt.hist(randos_peaks['X_Peak_Coordinate'],bins=bins,label="2000 Random Comparisons",alpha=0.3)
plt.hist(random_83_peaks['X_Peak_Coordinate'],bins=bins,label="83 Random Individuals > 100 EHRs",alpha=0.3)
plt.legend()

# plt.plot(randos_peaks['X_Peak_Coordinate'])
# plt.plot(random_83_peaks['X_Peak_Coordinate'])
control_group_coordinate_peaks

# Below is Extraneous Graph Code

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

plt.figure(figsize=(20,14))

# coefficients = []
comparison_functions = {}

i = -1
for dataset in result:
    i += 1
    x = np.array(list(dataset.keys()))
    y = np.array(list(dataset.values()))

    # This sucker is for finding the last nonzero value
    index = 0
    while index<len(y) and y[index]==0:
        index+=1


    y_cutoff = y[index:]
    x_cutoff = x[index:]
    if x_cutoff.size == 0:
#         comparison_functions[comparison_ids[i]] = None
        continue
#     print(x_cutoff)
    coefficients = np.polyfit(x_cutoff,y_cutoff,2)
    comparison_functions[comparison_ids[i]] = coefficients
#     print(coefficients)

    quadratic_model = np.poly1d(coefficients)
    # maybe not x cutoff?
    y_pred = quadratic_model(x_cutoff)

    x_fit = np.linspace(x.min(), x.max(), 100)
    y_fit = quadratic_model(x_fit)
#     if tuple(coefficients)[0] > 8:

# RIGHT HERE: ONLY PLOTTING TOP QUARTILE SLOPES
    if coefficients[0] > 5.5052460536468e-08:
#         possible_misdiagnosis[key] = value
        plt.plot(x_fit, y_fit)
plt.gca().invert_xaxis()
plt.xlabel('Days Before Cancer Diagnosis (Bins)', fontsize = 20)
plt.ylabel('Jaccard Score', fontsize = 20)
plt.title('Random Test Group: 83 Participants, EHRs > 100 - Pairwise Comparisons', fontsize = 25)
plt.xlim(0,-3000)
plt.ylim(-.05,.25)
plt.grid(True)

# plt.savefig("PairwiseComparisons_v2.pdf", format="pdf")
#plt.savefig("PairwiseComparisons.png")

plt.show()

In [ ]:
#control group comparisons
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Get all unique individuals in the dataframe
unique_people = filtered_binned['person_id'].unique().tolist()
num_individuals = 83

# Sample unique individuals
unique_people = filtered_binned['person_id'].sample(num_individuals, replace=False).tolist()

# Precompute bin-concept mappings for each person
person_bin_map = {
    person: {
        bin_: frozenset(group['concept_code'])  # Convert to frozenset for fast set operations
        for bin_, group in df.groupby('bin')
    }
    for person, df in filtered_binned.groupby('person_id') if person in unique_people
}

# Initialize similarity matrix
similarity_matrix_100x100 = np.eye(num_individuals)  # Identity matrix, 1s on diagonal

# Compare each pair only once
for i in range(num_individuals):
    person1 = unique_people[i]
    bins1 = person_bin_map[person1]

    for j in range(i + 1, num_individuals):
        person2 = unique_people[j]
        bins2 = person_bin_map[person2]

        # Compute average Jaccard similarity for all bin pairs
        similarity_scores = [
            len(bins1[b1] & bins2[b2]) / len(bins1[b1] | bins2[b2]) if len(bins1[b1] | bins2[b2]) > 0 else 0
            for b1 in bins1 for b2 in bins2
        ]

        # Take the average of all scores, or 0 if there are no valid comparisons
        avg_similarity_score = sum(similarity_scores) if similarity_scores else 0

        # Fill symmetric similarity matrix
        similarity_matrix_100x100[i, j] = avg_similarity_score
        similarity_matrix_100x100[j, i] = avg_similarity_score  # Mirror value

# ** Generate Heatmap **
plt.figure(figsize=(12, 8))

sns.clustermap(
    similarity_matrix_100x100,
    cmap="coolwarm",
    row_cluster=True,  # Cluster rows
    col_cluster=True,  # Cluster columns
    annot=False,
    xticklabels=False,
    yticklabels=False,
    figsize=(12, 8),
    cbar_kws={'label': "Jaccard Similarity Score"},

)

plt.title("Random Test Group -- Sum Jaccard Scores")
plt.xlabel("Individual Index")
plt.ylabel("Individual Index")

plt.show()

In [ ]:
#100 v 100 comparisons
from collections import OrderedDict
import statistics as st
import numpy as np
from itertools import combinations

fixed_people = binned_individuals['person_id'].drop_duplicates().sample(100, random_state=42).tolist()

indexes2actuallyplot = []
jscore_to_bin_dict = {}
matching_codes = []
people_compared = {}
people_bins_avgScores = {}
averages_with_bins = {}
split_avg = {}

for rando1, rando2 in combinations(fixed_people, 2):
    person1_df = binned_individuals[binned_individuals['person_id'] == rando1]
    person2_df = binned_individuals[binned_individuals['person_id'] == rando2]

    person1_bin_set = set(person1_df['bin'])
    person2_bin_set = set(person2_df['bin'])

    bin_pairs = np.array([(bin1, bin2) for bin1 in person1_bin_set for bin2 in person2_bin_set], dtype=object)

    Asub_list = {bin1: set(person1_df.loc[person1_df['bin'] == bin1, 'concept_code']) for bin1 in person1_bin_set}
    Bsub_list = {bin2: set(person2_df.loc[person2_df['bin'] == bin2, 'concept_code']) for bin2 in person2_bin_set}

    similarity_scores = np.array([
        len(Asub_list[bin1] & Bsub_list[bin2]) / len(Asub_list[bin1] | Bsub_list[bin2]) if len(Asub_list[bin1] | Bsub_list[bin2]) > 0 else 0.0
        for bin1, bin2 in bin_pairs
    ])

    similarity_matrix = similarity_scores.reshape(len(person1_bin_set), len(person2_bin_set))
    mask = np.ones(similarity_matrix.shape, dtype=bool)
    max_dict_ordered = OrderedDict()
    excluded_bins_to_raw_score = OrderedDict()

    while np.any(mask):
        masked_matrix = np.where(mask, similarity_matrix, -np.inf)
        max_index = np.argmax(masked_matrix)
        max_row, max_col = np.unravel_index(max_index, similarity_matrix.shape)
        max_score = similarity_matrix[max_row, max_col]

        tuple_key = tuple(bin_pairs[max_row * len(person2_bin_set) + max_col])
        max_dict_ordered[tuple_key] = max_score
        excluded_bins_to_raw_score.setdefault(tuple_key, []).append(max_score)

        mask[max_row, :] = False
        mask[:, max_col] = False

    sorted_ordered_dict = OrderedDict(sorted(max_dict_ordered.items(), key=lambda x: x[1], reverse=True))
    indexes2actuallyplot.append(list(sorted_ordered_dict.keys()))
    averages_to_actually_graph = [st.mean(sorted_ordered_dict.values())]

    sorted_bins_to_max_score_dict = OrderedDict(sorted(excluded_bins_to_raw_score.items(), key=lambda x: max(x[1]), reverse=True))
    while sorted_bins_to_max_score_dict:
        bins_from_matrix = list(sorted_bins_to_max_score_dict.keys())
        maxes_average = st.mean([val for values in sorted_bins_to_max_score_dict.values() for val in values])
        averages_with_bins.setdefault((rando1, rando2), []).append({bins_from_matrix[-1]: maxes_average})
        sorted_bins_to_max_score_dict.popitem()

    person1_best_scores = {key[0]: value[0] for key, value in excluded_bins_to_raw_score.items()}
    person2_best_scores = {key[1]: value[0] for key, value in excluded_bins_to_raw_score.items()}

    sorted_by_bins_person1 = OrderedDict(sorted(person1_best_scores.items()))
    sorted_by_bins_person2 = OrderedDict(sorted(person2_best_scores.items()))

    if (rando1, rando2) not in split_avg:
        split_avg[(rando1, rando2)] = []

    while sorted_by_bins_person2:
        closest_to_diagnosis_p2 = next(iter(sorted_by_bins_person2))
        curr_average = st.mean(sorted_by_bins_person2.values())
        split_avg[(rando1, rando2)].append({closest_to_diagnosis_p2: curr_average})
        del sorted_by_bins_person2[closest_to_diagnosis_p2]

print(split_avg)

In [ ]:
perfect_people = {1065536, 2466433, 2971750, 1864936, 1462411, 3401488, 1476979, 3504853, 2853368, 1793118, 1969183}
overall_avgs = {}
for pair in split_avg:
    total = 0
    count = 0
    for item in split_avg[pair]:
        for key, value in item.items():
            total += value
            count += 1
    if pair[0] not in perfect_people and pair[1] not in perfect_people:
        overall_avgs[pair] = total / count
print(overall_avgs)

In [ ]:
perfect_people = set()
for pair, avg_score in overall_avgs.items():
    if avg_score == 1:
        perfect_people.add(pair[0])
        perfect_people.add(pair[1])
print(perfect_people)

In [ ]:
# Create a 100x100 similarity matrix
num_individuals = 100
unique_people = list(set([p for pair in overall_avgs.keys() for p in pair]))  # Extract unique people
person_index_map = {person: idx for idx, person in enumerate(unique_people)}  # Mapping person_id -> matrix index

# Initialize similarity matrix
similarity_matrix_100x100 = np.eye(num_individuals)  # Identity matrix, 1s on diagonal

# Fill the matrix with averaged scores
for (person1, person2), avg_score in overall_avgs.items():
    i, j = person_index_map[person1], person_index_map[person2]
    similarity_matrix_100x100[i, j] = avg_score
    similarity_matrix_100x100[j, i] = avg_score  # Mirror it for symmetry


# Ensure zero is the lowest value and control how fast it turns red
vmin = 0  # Force blue at the lowest similarity
vmax = max(overall_avgs.values())  # Adjust the highest value dynamically
midpoint = vmax * 0.2  # Shift the transition point to red earlier

# Generate the Cluster Heatmap
plt.figure(figsize=(12, 8))
sns.clustermap(
    similarity_matrix_100x100,
    cmap="coolwarm",
    row_cluster=True,
    col_cluster=True,
    annot=False,
    xticklabels=False,
    yticklabels=False,
    figsize=(12, 8),
    cbar_kws={'label': "Rescaled Jaccard Similarity"},
    vmin=vmin,
    vmax=vmax,
    center=midpoint  # Shift midpoint so that red appears sooner
)

plt.title("Pairwise Similarity Heatmap (Enhanced Contrast)")
plt.xlabel("Individual Index")
plt.ylabel("Individual Index")
plt.show()

In [ ]:
#DO NOT RUN!!!!!! 2000 comparisons
from collections import OrderedDict
import statistics as st
import numpy as np
multiple_links = []
indexes2actuallyplot = []
jscore_to_bin_dict = {}
matching_codes = []
# hypertension_bins = []
# hyperlipidemia_bins = []
# e_hypertension_bins = []
# hypercholesterolema_bins = []
# diabetes_bins = []
# benign_bins = []
people_compared = {}
people_bins_avgScores = {}
averages_with_bins = {}
split_avg = {}

for _ in range(1000):
    rando1 = binned_individuals['person_id'].sample().iloc[0]
    rando2 = binned_individuals['person_id'].sample().iloc[0]
    # Filter DataFrame for rows belonging to person 1
    person1_df = binned_individuals[binned_individuals['person_id'] == rando1]
    # Extract the 'bin' column values into a list
    bin_values_person1 = person1_df['bin'].tolist()
    person1_bin_set = set(bin_values_person1)

    # Filter DataFrame for rows belonging to person 2
    person2_df = binned_individuals[binned_individuals['person_id'] == rando2]
    # Extract the 'bin' column values into a list
    bin_values_person2 = person2_df['bin'].tolist()
    person2_bin_set = set(bin_values_person2)

    similarity_scores_matrix = []
    bins_compared = []

    for bin1 in person1_bin_set:
        bin_scores = []
        bin_comparisons = []
        for bin2 in person2_bin_set:
            Asub = set(person1_df.loc[(person1_df['bin'] == bin1), 'condition_source_value'])
            Bsub = set(person2_df.loc[(person2_df['bin'] == bin2), 'condition_source_value'])
            C = Asub.intersection(Bsub)
            D = Asub.union(Bsub)
            if len(D) == 0:
                jsim = 0.0
            else:
                jsim = float(len(C))/float(len(D))
#             if len(C) > 0:
#                 matching_codes.append(C)
#                 for item in C:

#                     # search for benign ICDs (D for ICD 10, 2+ 1 or 2 for ICD 9)
#                     if item[0] == 'D':
#                         benign_bins.append(bin1)
#                         benign_bins.append(bin2)
#                     elif item[0] == '2':
#                         if item[1] == '1':
#                             benign_bins.append(bin1)
#                             benign_bins.append(bin2)
#                         elif item[1] == '2':
#                             benign_bins.append(bin1)
#                             benign_bins.append(bin2)

#                     # searching for conditions in ICD 9
#                     elif item == '401.9':
#                         hypertension_bins.append(bin1)
#                         hypertension_bins.append(bin2)
#                     elif item == '272.4':
#                         hyperlipidemia_bins.append(bin1)
#                         hyperlipidemia_bins.append(bin2)

#                     #searching for conditions in ICD 10
#                     elif item == 'I10':
#                         e_hypertension_bins.append(bin1)
#                         e_hypertension_bins.append(bin2)

#                     #ICD 9

#                     elif item == '272.0':
#                         hypercholesterolema_bins.append(bin1)
#                         hypercholesterolema_bins.append(bin2)
#                     elif item == '250.00':
#                         diabetes_bins.append(bin1)
#                         diabetes_bins.append(bin2)

            # search for significant similarity in Jscore (significance arb set as 0.1 for now)

            if (rando1, rando2) not in people_compared:
                people_compared[(rando1, rando2)] = []
            people_compared[(rando1, rando2)].append({(bin1, bin2) : jsim})
            bin_scores.append(jsim)
            bin_comparisons.append((bin1, bin2))
            jscore_to_bin_dict[jsim] = (bin1, bin2)
        similarity_scores_matrix.append(bin_scores)
        bins_compared.append(bin_comparisons)
    #row indexes correspond to indexes of person1_bin_set, column indexes correspond to person2_bin_set indexes

    #part 2:
    max_value = 0.0
    max_row = 0
    max_column = 0
    exclude_rows = []
    exclude_columns = []
    max_links = []
    maxes = []
    excluded_bins = []
    max_dict_ordered = OrderedDict()
    excluded_bins_to_raw_score = OrderedDict()

    #FOR REVIEW FOR MEM AND TIME EFFICIENCY
    #finds the max value in the whole raw matrix
    for row in range(len(similarity_scores_matrix)):
        for column in range(len(similarity_scores_matrix[0])):
            if similarity_scores_matrix[row][column] > max_value:
                max_value = similarity_scores_matrix[row][column]
                max_row = row
                max_column = column
                if bins_compared[row][column] not in excluded_bins_to_raw_score:
                    excluded_bins_to_raw_score[bins_compared[row][column]] = []
                excluded_bins_to_raw_score[bins_compared[row][column]].append(similarity_scores_matrix[row][column])
            else:
                continue
    exclude_rows.append(max_row)
    exclude_columns.append(max_column)
    maxes.append(max_value)
    #make double key ordered dictionary with the keys as row/column and the value as the max_value
    max_dict_ordered[(max_row, max_column)] = max_value

    #part 3
    avg_scores = []

    # Convert the similarity scores to a numpy array
    similarity_scores_np = np.array(similarity_scores_matrix)

    # Calculate the average using numpy.mean()
    overall_average_value = np.mean(similarity_scores_np)

    avg_scores.append(overall_average_value)
    while max_value > 0:
        new_scores = []
        new_bins = []
        for row in range(len(similarity_scores_matrix)):
            row_scores = []
            row_bins = []
            for column in range(len(similarity_scores_matrix[0])):
                if row in exclude_rows:
                    continue
                elif column in exclude_columns:
                    continue
                else:
                    row_scores.append(similarity_scores_matrix[row][column])
                    row_bins.append(bins_compared[row][column])
            new_scores.append(row_scores)
            new_bins.append(row_bins)

        # finding the max value
        max_value = 0.0
        max_row = 0
        max_column = 0
        for row in range(len(new_scores)):
            if len(new_scores[row]) == 0:
                continue
            for column in range(len(new_scores[row])):
                if new_scores[row][column] > max_value:
                    max_value = new_scores[row][column]
                    max_row = row
                    max_column = column
                    if new_bins[row][column] not in excluded_bins_to_raw_score:
                        excluded_bins_to_raw_score[new_bins[row][column]] = []
                    excluded_bins_to_raw_score[new_bins[row][column]].append(new_scores[row][column])
                else:
                    continue
        exclude_rows.append(max_row)
        exclude_columns.append(max_column)
        maxes.append(max_value)
        if len(new_bins[0]) > 0:
            excluded_bins.append(new_bins[max_row][max_column])
        #add indexes and max_value to max_links dictionary
        max_dict_ordered[(max_row, max_column)] = max_value

        # Convert the similarity scores to a numpy array
        build_array = []
        for row in new_scores:
            if len(row) > 0:
                build_array.append(row)
        new_scores_np = np.array(build_array)

        # Calculate the average using numpy.mean()
        avg_value = np.mean(new_scores_np)

        avg_scores.append(avg_value)
        if (rando1, rando2) not in people_bins_avgScores:
                people_bins_avgScores[(rando1, rando2)] = [overall_average_value]
        if len(new_bins[0]) > 0:
            people_bins_avgScores[(rando1, rando2)].append({new_bins[max_row][max_column] : avg_value})

    #part 4
    # Sort the keys
    sorted_keys = sorted(max_dict_ordered.keys(), reverse=True)

    # Create a new ordered dictionary with sorted keys
    sorted_ordered_dict = OrderedDict((key, max_dict_ordered[key]) for key in sorted_keys)
    links_to_graph = []
    for key1, value in sorted_ordered_dict.items():
        links_to_graph.append(value)

    averages_to_actually_graph = []
    # introduce an expand
    # eliminate
    # x is the bin we took out, and y is the average score

    while sorted_ordered_dict:
        avg_2_plot = [sorted_ordered_dict[key] for key in sorted_ordered_dict]
        averages_to_actually_graph.append(st.mean(avg_2_plot))
        indexes2use = [key for key in sorted_ordered_dict]
        indexes2actuallyplot.append(indexes2use)
        sorted_ordered_dict.popitem()

    sorted_keys = sorted(excluded_bins_to_raw_score.keys(), reverse=True)
    sorted_bins_to_max_score_dict = OrderedDict((key, excluded_bins_to_raw_score[key]) for key in sorted_keys)
    while sorted_bins_to_max_score_dict:
        maxes_from_matrix = [item for key in sorted_bins_to_max_score_dict for item in sorted_bins_to_max_score_dict[key]]
        bins_from_matrix = [key for key in sorted_bins_to_max_score_dict]
        maxes_average = st.mean(maxes_from_matrix)
        if (rando1, rando2) not in averages_with_bins:
            averages_with_bins[(rando1, rando2)] = []
        averages_with_bins[(rando1, rando2)].append({bins_from_matrix[-1] : maxes_average})
        sorted_bins_to_max_score_dict.popitem()
    person1_best_scores = {}
    person2_best_scores = {}
    #splitting the dictionary
    for key, value in excluded_bins_to_raw_score.items():
        person1_best_scores[key[0]] = value[0]
        person2_best_scores[key[1]] = value[0]
    sorted_by_bins_person1 = dict(sorted(person1_best_scores.items()))
    sorted_by_bins_person2 = dict(sorted(person2_best_scores.items()))
    if (rando1, rando2) not in split_avg:
        split_avg[(rando1, rando2)] = []
    curr_average = 0
    while sorted_by_bins_person2:
        closest_to_diagnosis_p2 = min(sorted_by_bins_person2)
        values = sorted_by_bins_person2.values()
        total = sum(values)
        curr_average = total / len(values)
        split_avg[(rando1, rando2)].append({closest_to_diagnosis_p2:curr_average})
        sorted_by_bins_person2.pop(closest_to_diagnosis_p2)

#     new_averages = []
#     y_intercept = averages_to_actually_graph[0]
#     for item in averages_to_actually_graph:
#         new_averages.append(item - y_intercept)
#     multiple_links.append(new_averages)

In [ ]:
print(split_avg)

In [ ]:
result = []
comparison_ids = [] # This is storing tuples
# the key is the person, the person is the dictionary of bins to scores
for key, value in split_avg.items():
    comparison_ids.append(key)
    person2_bins = {}
    for val in value:
        for k, v in val.items():
            bin = int(k.split(',')[0])
            person2_bins[bin] = v
    scores = {}
    score_list = []
    curr_score = 0
    for j in bins:
            curr_bin = j
            if j not in person2_bins:
                curr_score = curr_score
            else:
                curr_score = person2_bins[j]
            scores[curr_bin] = curr_score
            score_list.append(curr_score)
    max_score = 0
    for item in score_list:
        if item > max_score:
            max_score = item
    if max_score != 1.0:
        result.append(scores)
print(result)

In [ ]:
import matplotlib.pyplot as plt
# Extract x-values (assuming they are the same across dictionaries)
x_values = list(result[0].keys())

# Extract y-values for each dictionary
y_values_list = [list(d.values()) for d in result]

# Plot each dataset
for y_values in y_values_list:
    plt.plot(x_values, y_values, marker='o', linestyle='-')

# Labels and title
plt.xlabel("X-axis (e.g., time)")
plt.ylabel("Y-axis (values)")
plt.title("Plot of Given Data")
plt.legend([f"Dataset {i+1}" for i in range(len(y_values_list))])

plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

plt.figure(figsize=(20,14))

# coefficients = []
comparison_functions = {}

i = -1
for dataset in result:
    i += 1
    x = np.array(list(dataset.keys()))
    y = np.array(list(dataset.values()))

    # This sucker is for finding the last nonzero value
    index = 0
    while index<len(y) and y[index]==0:
        index+=1


    y_cutoff = y[index:]
    x_cutoff = x[index:]
    if x_cutoff.size == 0:
#         comparison_functions[comparison_ids[i]] = None
        continue
#     print(x_cutoff)
    coefficients = np.polyfit(x_cutoff,y_cutoff,2)
    comparison_functions[comparison_ids[i]] = coefficients
#     print(coefficients)

    quadratic_model = np.poly1d(coefficients)
    # maybe not x cutoff?
    y_pred = quadratic_model(x_cutoff)

    x_fit = np.linspace(x.min(), x.max(), 100)
    y_fit = quadratic_model(x_fit)
#     if tuple(coefficients)[0] > 8:

# RIGHT HERE: ONLY PLOTTING TOP QUARTILE SLOPES
    if coefficients[0] > 5.5052460536468e-08:
#         possible_misdiagnosis[key] = value
        plt.plot(x_fit, y_fit)
plt.gca().invert_xaxis()
plt.xlabel('Days Before Cancer Diagnosis (Bins)', fontsize = 20)
plt.ylabel('Jaccard Score', fontsize = 20)
plt.title('2000 Pairwise Comparisons', fontsize = 25)
plt.xlim(0,-3000)
plt.ylim(-.05,.25)
plt.grid(True)

plt.savefig("PairwiseComparisons_v2.pdf", format="pdf")
#plt.savefig("PairwiseComparisons.png")



# plt.figure(figsize=(20,8))

# for dataset in result:
#     x = np.array(list(dataset.keys()))
#     y = np.array(list(dataset.values()))
#     coefficients = np.polyfit(x,y,2)
#     print(coefficients)


# #     print(y)
#     quadratic_model = np.poly1d(coefficients)
#     y_pred = quadratic_model(x)

#     x_fit = np.linspace(x.min(), x.max(), 100)
#     y_fit = quadratic_model(x_fit)
# #     print(y_fit)
#     mx = np.argmax(y_fit)
#     if coefficients[0] > 0:
#         mx = np.argmin(y_fit)

#     print(f"Quadratic Model local Min/Max = {x[mx]}")
#     print()
#     plt.plot(x_fit, y_fit)


# plt.gca().invert_xaxis()
# # Add labels and title
# plt.xlabel('Bins -15000 -> 0')
# plt.ylabel('Jccard Score')
# plt.title('Person 2 Bins vs Score')

# # Display the graph
# plt.grid(True)
# plt.show()

In [ ]:
# This snippet assumes that you run setup first

# This code lists objects in your Google Bucket

# Get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# List objects in the bucket
print(subprocess.check_output(f"gsutil ls -r {my_bucket}", shell=True).decode('utf-8'))

In [ ]:
# This cell grabs the top 20 highest slopes, the people out of the 2000 randos that we'd like to investigate
top_20 = dict(sorted(comparison_functions.items(), key=lambda item: item[1][0], reverse=True)[:20])
for key,value in top_20.items():
        print(f"Person IDs: {key}, Function = {value[0]}x^2 + {value[1]}x + {value[2]}")

In [ ]:
possible_misdiagnosis = {}
slope_coefficients = []

for key, value in comparison_functions.items():
    if value is None:
        continue
    if value[0] > 5.5052460536468e-09:
        possible_misdiagnosis[key] = value
    if value[0] > 0:
        slope_coefficients.append(value[0])

    if value[0]>6.0e-07:
        print(f"Person IDs: {key}, Function = {value[0]}x^2 + {value[1]}x + {value[2]}")


#     print(key)
#     print(value)

In [ ]:
# FINDING THE SUMMARY OF OUR COMPARISON LINES!

minimum = np.min(slope_coefficients)
q1 = np.percentile(slope_coefficients, 25)
median = np.median(slope_coefficients)
q3 = np.percentile(slope_coefficients, 75)
maximum = np.max(slope_coefficients)

print(f"""
Quadratic Coefficients 5-Number Summary
Minimum = {minimum}
First Quartile = {q1}
Median = {median}
Third Quartile = {q3}
Maximum = {maximum}
""")


for key, value in possible_misdiagnosis.items():
    print(f"Person IDs: {key}, Function = {value[0]}x^2 + {value[1]}x + {value[2]}")

In [ ]:
#DO NOT RUN THIS ONE!!!!!!! 1 to 100

from collections import OrderedDict
import statistics as st
import numpy as np
multiple_links = []
indexes2actuallyplot = []
jscore_to_bin_dict = {}
matching_codes = []
people_compared = {}
people_bins_avgScores = {}
averages_with_bins = {}
split_avg = {}

rando2 = 1604065#8654106#binned_individuals['person_id'].sample().iloc[0]

# EDITED HERE TO ADD A LOOP TO GO THROUGH EACH PERSON IN OUR TOP 20 SLOPES!!
for person_pair in top_20:
    rando2 = person_pair[0]
    # REMOVE THIS LOOP IF YOU WANT TO DO JUST 1 PERSON

    for i in range(100):
        rando1 = binned_individuals['person_id'].sample().iloc[0]
        # Filter DataFrame for rows belonging to person 1
        person1_df = binned_individuals[binned_individuals['person_id'] == rando1]
        # Extract the 'bin' column values into a list
        bin_values_person1 = person1_df['bin'].tolist()
        person1_bin_set = set(bin_values_person1)

        # Filter DataFrame for rows belonging to person 2
        person2_df = binned_individuals[binned_individuals['person_id'] == rando2]
        # Extract the 'bin' column values into a list
        bin_values_person2 = person2_df['bin'].tolist()
        person2_bin_set = set(bin_values_person2)

        similarity_scores_matrix = []
        bins_compared = []

        for bin1 in person1_bin_set:
            bin_scores = []
            bin_comparisons = []
            for bin2 in person2_bin_set:
                Asub = set(person1_df.loc[(person1_df['bin'] == bin1), 'condition_source_value'])
                Bsub = set(person2_df.loc[(person2_df['bin'] == bin2), 'condition_source_value'])
                C = Asub.intersection(Bsub)
                D = Asub.union(Bsub)
                if len(D) == 0:
                    jsim = 0.0
                else:
                    jsim = float(len(C))/float(len(D))
                if (rando1, rando2) not in people_compared:
                    people_compared[(rando1, rando2)] = []
                people_compared[(rando1, rando2)].append({(bin1, bin2) : jsim})
                bin_scores.append(jsim)
                bin_comparisons.append((bin1, bin2))
                jscore_to_bin_dict[jsim] = (bin1, bin2)
            similarity_scores_matrix.append(bin_scores)
            bins_compared.append(bin_comparisons)
        #row indexes correspond to indexes of person1_bin_set, column indexes correspond to person2_bin_set indexes

        #part 2:
        max_value = 0.0
        max_row = 0
        max_column = 0
        exclude_rows = []
        exclude_columns = []
        max_links = []
        maxes = []
        excluded_bins = []
        max_dict_ordered = OrderedDict()
        excluded_bins_to_raw_score = OrderedDict()

        #finds the max value in the whole raw matrix
        for row in range(len(similarity_scores_matrix)):
            for column in range(len(similarity_scores_matrix[0])):
                if similarity_scores_matrix[row][column] > max_value:
                    max_value = similarity_scores_matrix[row][column]
                    max_row = row
                    max_column = column
                    if bins_compared[row][column] not in excluded_bins_to_raw_score:
                        excluded_bins_to_raw_score[bins_compared[row][column]] = []
                    excluded_bins_to_raw_score[bins_compared[row][column]].append(similarity_scores_matrix[row][column])
                else:
                    continue
        exclude_rows.append(max_row)
        exclude_columns.append(max_column)
        maxes.append(max_value)
        #make double key ordered dictionary with the keys as row/column and the value as the max_value
        max_dict_ordered[(max_row, max_column)] = max_value

        #part 3
        avg_scores = []

        # Convert the similarity scores to a numpy array
        similarity_scores_np = np.array(similarity_scores_matrix)

        # Calculate the average using numpy.mean()
        overall_average_value = np.mean(similarity_scores_np)

        avg_scores.append(overall_average_value)
        while max_value > 0:
            new_scores = []
            new_bins = []
            for row in range(len(similarity_scores_matrix)):
                row_scores = []
                row_bins = []
                for column in range(len(similarity_scores_matrix[0])):
                    if row in exclude_rows:
                        continue
                    elif column in exclude_columns:
                        continue
                    else:
                        row_scores.append(similarity_scores_matrix[row][column])
                        row_bins.append(bins_compared[row][column])
                new_scores.append(row_scores)
                new_bins.append(row_bins)

            # finding the max value
            max_value = 0.0
            max_row = 0
            max_column = 0
            for row in range(len(new_scores)):
                if len(new_scores[row]) == 0:
                    continue
                for column in range(len(new_scores[row])):
                    if new_scores[row][column] > max_value:
                        max_value = new_scores[row][column]
                        max_row = row
                        max_column = column
                        if new_bins[row][column] not in excluded_bins_to_raw_score:
                            excluded_bins_to_raw_score[new_bins[row][column]] = []
                        excluded_bins_to_raw_score[new_bins[row][column]].append(new_scores[row][column])
                    else:
                        continue
            exclude_rows.append(max_row)
            exclude_columns.append(max_column)
            maxes.append(max_value)
            if len(new_bins[0]) > 0:
                excluded_bins.append(new_bins[max_row][max_column])
            #add indexes and max_value to max_links dictionary
            max_dict_ordered[(max_row, max_column)] = max_value

            # Convert the similarity scores to a numpy array
            build_array = []
            for row in new_scores:
                if len(row) > 0:
                    build_array.append(row)
            new_scores_np = np.array(build_array)

            # Calculate the average using numpy.mean()
            avg_value = np.mean(new_scores_np)

            avg_scores.append(avg_value)
            if (rando1, rando2) not in people_bins_avgScores:
                    people_bins_avgScores[(rando1, rando2)] = [overall_average_value]
            if len(new_bins[0]) > 0:
                people_bins_avgScores[(rando1, rando2)].append({new_bins[max_row][max_column] : avg_value})

        #part 4
        # Sort the keys
        sorted_keys = sorted(max_dict_ordered.keys(), reverse=True)

        # Create a new ordered dictionary with sorted keys
        sorted_ordered_dict = OrderedDict((key, max_dict_ordered[key]) for key in sorted_keys)
        links_to_graph = []
        for key1, value in sorted_ordered_dict.items():
            links_to_graph.append(value)

        averages_to_actually_graph = []
        # introduce an expand
        # eliminate
        # x is the bin we took out, and y is the average score

        while sorted_ordered_dict:
            avg_2_plot = [sorted_ordered_dict[key] for key in sorted_ordered_dict]
            averages_to_actually_graph.append(st.mean(avg_2_plot))
            indexes2use = [key for key in sorted_ordered_dict]
            indexes2actuallyplot.append(indexes2use)
            sorted_ordered_dict.popitem()

        sorted_keys = sorted(excluded_bins_to_raw_score.keys(), reverse=True)
        sorted_bins_to_max_score_dict = OrderedDict((key, excluded_bins_to_raw_score[key]) for key in sorted_keys)
        while sorted_bins_to_max_score_dict:
            maxes_from_matrix = [item for key in sorted_bins_to_max_score_dict for item in sorted_bins_to_max_score_dict[key]]
            bins_from_matrix = [key for key in sorted_bins_to_max_score_dict]
            maxes_average = st.mean(maxes_from_matrix)
            if (rando1, rando2) not in averages_with_bins:
                averages_with_bins[(rando1, rando2)] = []
            averages_with_bins[(rando1, rando2)].append({bins_from_matrix[-1] : maxes_average})
            sorted_bins_to_max_score_dict.popitem()
        person1_best_scores = {}
        person2_best_scores = {}
        #splitting the dictionary
        for key, value in excluded_bins_to_raw_score.items():
            person1_best_scores[key[0]] = value[0]
            person2_best_scores[key[1]] = value[0]
        sorted_by_bins_person1 = dict(sorted(person1_best_scores.items()))
        sorted_by_bins_person2 = dict(sorted(person2_best_scores.items()))
        if (rando1, rando2) not in split_avg:
            split_avg[(rando1, rando2)] = []
        curr_average = 0
        while sorted_by_bins_person2:
            closest_to_diagnosis_p2 = min(sorted_by_bins_person2)
            values = sorted_by_bins_person2.values()
            total = sum(values)
            curr_average = total / len(values)
            split_avg[(rando1, rando2)].append({closest_to_diagnosis_p2:curr_average})
            sorted_by_bins_person2.pop(closest_to_diagnosis_p2)

In [ ]:
print(split_avg)

In [ ]:
result = []
comparison_ids = []
for key, value in split_avg.items():
    comparison_ids.append(key)
    person2_bins = {}
    for val in value:
        for k, v in val.items():
            bin = int(k.split(',')[0])
            person2_bins[bin] = v
    scores = {}
    score_list = []
    curr_score = 0
    for j in bins:
            curr_bin = j
            if j not in person2_bins:
                curr_score = curr_score
            else:
                curr_score = person2_bins[j]
            scores[curr_bin] = curr_score
            score_list.append(curr_score)
    max_score = 0
    for item in score_list:
        if item > max_score:
            max_score = item
    if max_score != 1.0:
        result.append(scores)

In [ ]:
# import matplotlib.pyplot as plt
# from sklearn.preprocessing import PolynomialFeatures
# from sklearn.linear_model import LinearRegression
# import matplotlib.pyplot as plt

# plt.figure(figsize=(20,14))

# # coefficients = []
# comparison_functions = {}

# i = -1
# for dataset in result:
#     i += 1
#     x = np.array(list(dataset.keys()))
#     y = np.array(list(dataset.values()))

#     # This sucker is for finding the last nonzero value
#     index = 0
#     while index<len(y) and y[index]==0:
#         index+=1


#     y_cutoff = y[index:]
#     x_cutoff = x[index:]
#     if x_cutoff.size == 0:
#         comparison_functions[comparison_ids[i]] = None
#         continue
# #     print(x_cutoff)
#     coefficients = np.polyfit(x_cutoff,y_cutoff,2)
#     comparison_functions[comparison_ids[i]] = coefficients
# #     print(coefficients)

#     quadratic_model = np.poly1d(coefficients)
#     # maybe not x cutoff?
#     y_pred = quadratic_model(x_cutoff)

#     x_fit = np.linspace(x.min(), x.max(), 100)
#     y_fit = quadratic_model(x_fit)
# #     if tuple(coefficients)[0] > 8:

# # RIGHT HERE: ONLY PLOTTING TOP QUARTILE SLOPES
#     if coefficients[0] > 5.5052460536468e-08:
# #         possible_misdiagnosis[key] = value
#         plt.plot(x_fit, y_fit)
# plt.gca().invert_xaxis()
# plt.xlabel('Bins -15000 -> 0')
# plt.ylabel('Jccard Score')
# plt.title('Person 2 Bins vs Score')
# plt.xlim(0,-3000)
# plt.ylim(-.25,.25)
# plt.grid(True)
# plt.show()


# possible_misdiagnosis = {}
# slope_coefficients = []

# for key, value in comparison_functions.items():
#     if value is None:
#         continue
#     if value[0] > 5.5052460536468e-09:
#         possible_misdiagnosis[key] = value
#     if value[0] > 0:
#         slope_coefficients.append(value[0])

#     if value[0]>6.0e-07:
#         print(f"Person IDs: {key}, Function = {value[0]}x^2 + {value[1]}x + {value[2]}")


# #     print(key)
# #     print(value)

In [ ]:
#DON't RUN THESE GUYS !!!!!!!
import matplotlib.pyplot as plt

plt.figure(figsize=(20,8))

last_nonzero_y_index = 0
last_nonzero_y_indices = []
index_nonzero = 150




# for dataset in result:
#     i = 0
#     while list(dataset.values())[i] != 0:
#         last_nonzero_y_indices.append(i)



#     for i in range(len(dataset.values())):
#         if list(dataset.values())[i] == 0 and i < index_nonzero:
#             last_nonzero_y_indices.append(i)
#             index_nonzero = i

# last_nonzero_y_index = max(last_nonzero_y_indices)
# print(last_nonzero_y_index)


for dataset in result:
    x = list(dataset.keys())#[:last_nonzero_y_index]
    y = list(dataset.values())#[:last_nonzero_y_index]
    plt.plot(x, y)
plt.gca().invert_xaxis()
# Add labels and title

plt.xlim(0,-2500)
plt.xlabel('Bins -15000 -> 0')
plt.ylabel('Jccard Score')
plt.title(f'{rando2} Bins vs Score')

# Display the graph
plt.grid(True)
plt.show()

In [ ]:
#### DON't RUN THIS

import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

plt.figure(figsize=(20,14))

# coefficients = []
comparison_functions = {}

i = -1
for dataset in result:
    i += 1
    x = np.array(list(dataset.keys()))
    y = np.array(list(dataset.values()))

    # This sucker is for finding the last nonzero value
    index = 0
    while index<len(y) and y[index]==0:
        index+=1


    y_cutoff = y[index:]
    x_cutoff = x[index:]
    if x_cutoff.size == 0:
        comparison_functions[comparison_ids[i]] = None
        continue
#     print(x_cutoff)
    coefficients = np.polyfit(x_cutoff,y_cutoff,2)
    comparison_functions[comparison_ids[i]] = coefficients
#     print(coefficients)

    quadratic_model = np.poly1d(coefficients)
    # maybe not x cutoff?
    y_pred = quadratic_model(x_cutoff)

    x_fit = np.linspace(x.min(), x.max(), 100)
    y_fit = quadratic_model(x_fit)
#     if tuple(coefficients)[0] > 8:

# RIGHT HERE: ONLY PLOTTING TOP QUARTILE SLOPES
    if (coefficients[0]) < -7.5052460536468e-09:
        possible_misdiagnosis[key] = value
        plt.plot(x_fit, y_fit)
plt.gca().invert_xaxis()
plt.xlabel('Bins -15000 -> 0')
plt.ylabel('Jccard Score')
plt.title('Person 2 Bins vs Score')
plt.xlim(0,-3000)
plt.ylim(-.25,.25)
plt.grid(True)
plt.show()


possible_misdiagnosis = {}
slope_coefficients = []

for key, value in comparison_functions.items():
    if value is None:
        continue
    if abs(value[0]) < -7.5052460536468e-09:
        possible_misdiagnosis[key] = value
    if value[0] > 0:
        slope_coefficients.append(value[0])

#     if value[0]>6.0e-07:
#         print(f"Person IDs: {key}, Function = {value[0]}x^2 + {value[1]}x + {value[2]}")


#     print(key)
#     print(value)

print(possible_misdiagnosis)

In [ ]:
#DON't RUN tHIS
print(averages_with_bins)

In [ ]:
#DON'T RUN THIS####
result = []
for key, value in averages_with_bins.items():
    person1_bins = {}
    for val in value:
        for k, v in val.items():
            bin = int(k[0].split(',')[0])
            person1_bins[bin] = v
    scores = {}
    curr_score = 0
    for j in bins:
            curr_bin = j
            if j not in person1_bins:
                curr_score = curr_score
            else:
                curr_score = person1_bins[j]
            scores[curr_bin] = curr_score
    result.append(scores)
print(result)

# 20-person Figure Development.

In [ ]:
#RUN THIS
#1 to 100

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
from collections import OrderedDict
import statistics as st
import numpy as np
multiple_links = []
indexes2actuallyplot = []
jscore_to_bin_dict = {}
matching_codes = []
people_compared = {}
people_bins_avgScores = {}
averages_with_bins = {}
split_avg = {}

each_person_average_quad_function = {}


rando2 = 1604065#8654106#binned_individuals['person_id'].sample().iloc[0]

# EDITED HERE TO ADD A LOOP TO GO THROUGH EACH PERSON IN OUR TOP 20 SLOPES!!
for person_pair in top_20:
    rando2 = person_pair[0]
    # REMOVE THIS LOOP IF YOU WANT TO DO JUST 1 PERSON
    for i in range(100):
        rando1 = binned_individuals['person_id'].sample().iloc[0]
        # Filter DataFrame for rows belonging to person 1
        person1_df = binned_individuals[binned_individuals['person_id'] == rando1]
        # Extract the 'bin' column values into a list
        bin_values_person1 = person1_df['bin'].tolist()
        person1_bin_set = set(bin_values_person1)

        # Filter DataFrame for rows belonging to person 2
        person2_df = binned_individuals[binned_individuals['person_id'] == rando2]
        # Extract the 'bin' column values into a list
        bin_values_person2 = person2_df['bin'].tolist()
        person2_bin_set = set(bin_values_person2)

        similarity_scores_matrix = []
        bins_compared = []

        for bin1 in person1_bin_set:
            bin_scores = []
            bin_comparisons = []
            for bin2 in person2_bin_set:
                Asub = set(person1_df.loc[(person1_df['bin'] == bin1), 'condition_source_value'])
                Bsub = set(person2_df.loc[(person2_df['bin'] == bin2), 'condition_source_value'])
                C = Asub.intersection(Bsub)
                D = Asub.union(Bsub)
                if len(D) == 0:
                    jsim = 0.0
                else:
                    jsim = float(len(C))/float(len(D))
                if (rando1, rando2) not in people_compared:
                    people_compared[(rando1, rando2)] = []
                people_compared[(rando1, rando2)].append({(bin1, bin2) : jsim})
                bin_scores.append(jsim)
                bin_comparisons.append((bin1, bin2))
                jscore_to_bin_dict[jsim] = (bin1, bin2)
            similarity_scores_matrix.append(bin_scores)
            bins_compared.append(bin_comparisons)
        #row indexes correspond to indexes of person1_bin_set, column indexes correspond to person2_bin_set indexes

        #part 2:
        max_value = 0.0
        max_row = 0
        max_column = 0
        exclude_rows = []
        exclude_columns = []
        max_links = []
        maxes = []
        excluded_bins = []
        max_dict_ordered = OrderedDict()
        excluded_bins_to_raw_score = OrderedDict()

        #finds the max value in the whole raw matrix
        for row in range(len(similarity_scores_matrix)):
            for column in range(len(similarity_scores_matrix[0])):
                if similarity_scores_matrix[row][column] > max_value:
                    max_value = similarity_scores_matrix[row][column]
                    max_row = row
                    max_column = column
                    if bins_compared[row][column] not in excluded_bins_to_raw_score:
                        excluded_bins_to_raw_score[bins_compared[row][column]] = []
                    excluded_bins_to_raw_score[bins_compared[row][column]].append(similarity_scores_matrix[row][column])
                else:
                    continue
        exclude_rows.append(max_row)
        exclude_columns.append(max_column)
        maxes.append(max_value)
        #make double key ordered dictionary with the keys as row/column and the value as the max_value
        max_dict_ordered[(max_row, max_column)] = max_value

        #part 3
        avg_scores = []

        # Convert the similarity scores to a numpy array
        similarity_scores_np = np.array(similarity_scores_matrix)

        # Calculate the average using numpy.mean()
        overall_average_value = np.mean(similarity_scores_np)

        avg_scores.append(overall_average_value)
        while max_value > 0:
            new_scores = []
            new_bins = []
            for row in range(len(similarity_scores_matrix)):
                row_scores = []
                row_bins = []
                for column in range(len(similarity_scores_matrix[0])):
                    if row in exclude_rows:
                        continue
                    elif column in exclude_columns:
                        continue
                    else:
                        row_scores.append(similarity_scores_matrix[row][column])
                        row_bins.append(bins_compared[row][column])
                new_scores.append(row_scores)
                new_bins.append(row_bins)

            # finding the max value
            max_value = 0.0
            max_row = 0
            max_column = 0
            for row in range(len(new_scores)):
                if len(new_scores[row]) == 0:
                    continue
                for column in range(len(new_scores[row])):
                    if new_scores[row][column] > max_value:
                        max_value = new_scores[row][column]
                        max_row = row
                        max_column = column
                        if new_bins[row][column] not in excluded_bins_to_raw_score:
                            excluded_bins_to_raw_score[new_bins[row][column]] = []
                        excluded_bins_to_raw_score[new_bins[row][column]].append(new_scores[row][column])
                    else:
                        continue
            exclude_rows.append(max_row)
            exclude_columns.append(max_column)
            maxes.append(max_value)
            if len(new_bins[0]) > 0:
                excluded_bins.append(new_bins[max_row][max_column])
            #add indexes and max_value to max_links dictionary
            max_dict_ordered[(max_row, max_column)] = max_value

            # Convert the similarity scores to a numpy array
            build_array = []
            for row in new_scores:
                if len(row) > 0:
                    build_array.append(row)
            new_scores_np = np.array(build_array)

            # Calculate the average using numpy.mean()
            avg_value = np.mean(new_scores_np)

            avg_scores.append(avg_value)
            if (rando1, rando2) not in people_bins_avgScores:
                    people_bins_avgScores[(rando1, rando2)] = [overall_average_value]
            if len(new_bins[0]) > 0:
                people_bins_avgScores[(rando1, rando2)].append({new_bins[max_row][max_column] : avg_value})

        #part 4
        # Sort the keys
        sorted_keys = sorted(max_dict_ordered.keys(), reverse=True)

        # Create a new ordered dictionary with sorted keys
        sorted_ordered_dict = OrderedDict((key, max_dict_ordered[key]) for key in sorted_keys)
        links_to_graph = []
        for key1, value in sorted_ordered_dict.items():
            links_to_graph.append(value)

        averages_to_actually_graph = []
        # introduce an expand
        # eliminate
        # x is the bin we took out, and y is the average score

        while sorted_ordered_dict:
            avg_2_plot = [sorted_ordered_dict[key] for key in sorted_ordered_dict]
            averages_to_actually_graph.append(st.mean(avg_2_plot))
            indexes2use = [key for key in sorted_ordered_dict]
            indexes2actuallyplot.append(indexes2use)
            sorted_ordered_dict.popitem()

        sorted_keys = sorted(excluded_bins_to_raw_score.keys(), reverse=True)
        sorted_bins_to_max_score_dict = OrderedDict((key, excluded_bins_to_raw_score[key]) for key in sorted_keys)
        while sorted_bins_to_max_score_dict:
            maxes_from_matrix = [item for key in sorted_bins_to_max_score_dict for item in sorted_bins_to_max_score_dict[key]]
            bins_from_matrix = [key for key in sorted_bins_to_max_score_dict]
            maxes_average = st.mean(maxes_from_matrix)
            if (rando1, rando2) not in averages_with_bins:
                averages_with_bins[(rando1, rando2)] = []
            averages_with_bins[(rando1, rando2)].append({bins_from_matrix[-1] : maxes_average})
            sorted_bins_to_max_score_dict.popitem()
        person1_best_scores = {}
        person2_best_scores = {}
        #splitting the dictionary
        for key, value in excluded_bins_to_raw_score.items():
            person1_best_scores[key[0]] = value[0]
            person2_best_scores[key[1]] = value[0]
        sorted_by_bins_person1 = dict(sorted(person1_best_scores.items()))
        sorted_by_bins_person2 = dict(sorted(person2_best_scores.items()))
        if (rando1, rando2) not in split_avg:
            split_avg[(rando1, rando2)] = []
        curr_average = 0
        while sorted_by_bins_person2:
            closest_to_diagnosis_p2 = min(sorted_by_bins_person2)
            values = sorted_by_bins_person2.values()
            total = sum(values)
            curr_average = total / len(values)
            split_avg[(rando1, rando2)].append({closest_to_diagnosis_p2:curr_average})
            sorted_by_bins_person2.pop(closest_to_diagnosis_p2)
    result = []
    comparison_ids = []
    for key, value in split_avg.items():
        comparison_ids.append(key)
        person2_bins = {}
        for val in value:
            for k, v in val.items():
                bin = int(k.split(',')[0])
                person2_bins[bin] = v
        scores = {}
        score_list = []
        curr_score = 0
        for j in bins:
                curr_bin = j
                if j not in person2_bins:
                    curr_score = curr_score
                else:
                    curr_score = person2_bins[j]
                scores[curr_bin] = curr_score
                score_list.append(curr_score)
        max_score = 0
        for item in score_list:
            if item > max_score:
                max_score = item
        if max_score != 1.0:
            result.append(scores)
    a = 0
    b = 0
    c = 0
    comparison_functions = {}
    num_counted = 1
    i = -1
    for dataset in result:
        i += 1
        x = np.array(list(dataset.keys()))
        y = np.array(list(dataset.values()))

        # This sucker is for finding the last nonzero value
        index = 0
        while index<len(y) and y[index]==0:
            index+=1
        y_cutoff = y[index:]
        x_cutoff = x[index:]
        if x_cutoff.size == 0:
            comparison_functions[comparison_ids[i]] = None
            continue
    #     print(x_cutoff)
        coefficients = np.polyfit(x_cutoff,y_cutoff,2)
        if coefficients[0] < -7.5052460536468e-09:
            a += coefficients[0]
            b += coefficients[1]
            c += coefficients[2]
            num_counted += 1
            comparison_functions[comparison_ids[i]] = coefficients
    # HERE: Let's calculate the averages
    quadratic_model = np.poly1d((a/num_counted,b/num_counted,c/num_counted))
    y_pred = quadratic_model(x_cutoff)
    x_fit = np.linspace(x.min(), x.max(), 100)
    y_fit = quadratic_model(x_fit)
    #     if tuple(coefficients)[0] > 8:

    # RIGHT HERE: ONLY PLOTTING TOP QUARTILE SLOPES
#         if coefficients[0] > 5.5052460536468e-08:
    #         possible_misdiagnosis[key] = value
    plt.plot(x_fit, y_fit)
#     for key,value in comparison_functions:
    each_person_average_quad_function[person_pair[0]] = (a/num_counted,b/num_counted,c/num_counted)
plt.figure(figsize=(20,14))
plt.gca().invert_xaxis()
plt.xlabel('Days Before Cancer Diagnosis (Bins)', fontsize = 20)
plt.ylabel('Jaccard Score', fontsize = 20)
plt.title('Indication of Misdiagnosis', fontsize = 25)
plt.xlim(0,-3000)
plt.ylim(-.25,.25)
plt.grid(True)

plt.savefig("IndicationMisdiagnosis_v2.pdf", format="pdf")
#plt.savefig("IndicationMisdiagnosis.png")

print(each_person_average_quad_function)

In [ ]:
plt.figure(figsize=(20,14))
plt.gca().invert_xaxis()
plt.xlabel('Days Before Cancer Diagnosis (Bins)', fontsize = 20)
plt.ylabel('Jaccard Score', fontsize = 20)
plt.title('Indication of Misdiagnosis', fontsize = 25)
plt.xlim(0,-3000)
plt.ylim(-.25,.25)
plt.grid(True)
plot.show()
plt.savefig("IndicationMisdiagnosis_v2.pdf", format="pdf")

In [ ]:
#DON't RUN THIS####
for key, value in each_person_average_quad_function.items():
    if value is None:
        continue
    x_fit = np.linspace(-15000, 0, 100)
    quadratic_model = np.poly1d(value)
    y_fit = quadratic_model(x_fit)
    plt.plot(x_fit, y_fit)
    print(f"Person ID: {key}, Function = {value[0]}x^2 + {value[1]}x + {value[2]}")
# plt.figure(figsize=(20,14))
plt.gca().invert_xaxis()
plt.xlabel('Days Before Cancer Diagnosis (Bins)', fontsize = 10)
plt.ylabel('Jaccard Score', fontsize = 10)
plt.title('Indication of Misdiagnosis', fontsize = 20)
plt.xlim(0,-2500)
plt.ylim(-.1,.25)
plt.grid(True)
plt.show()
print(len(each_person_average_quad_function))

In [ ]:
#SKIP
import matplotlib.pyplot as plt

for dataset in result:
    x = list(dataset.keys())
    y = list(dataset.values())
    plt.plot(x, y)

# Add labels and title
plt.xlabel('Bins -15000 -> 0')
plt.ylabel('Jccard Score')
plt.title('Person 1 Bins vs Score')

# Display the graph
plt.grid(True)
plt.show()

In [ ]:
#SKIP
# print(people_bins_avgScores)

In [ ]:
result = []
for key, value in people_bins_avgScores.items():
    person1_bins = {}
    for val in value:
        for k, v in val.items():
            bin = int(k[0].split(',')[0])
            person1_bins[bin] = v
    scores = {}
    curr_score = 0
    for j in bins:
            curr_bin = j
            if j not in person1_bins:
                curr_score = curr_score
            else:
                curr_score = person1_bins[j]
            scores[curr_bin] = curr_score
    result.append(scores)
print(result)

In [ ]:
import matplotlib.pyplot as plt

for dataset in result:
    x = list(dataset.keys())
    y = list(dataset.values())
    plt.plot(x, y)

# Add labels and title
plt.xlabel('Bins -15000 -> 0')
plt.ylabel('Jccard Score')
plt.title('Person 1 Bins vs Score')

# Display the graph
plt.grid(True)
plt.show()

In [ ]:
print(people_compared)

In [ ]:
result = []
for key, value in people_bins_avgScores.items():
    person1_bins = {}
    for val in value:
        for k, v in val.items():
            bin = int(k[0].split(',')[0])
            person1_bins[bin] = v
    scores = {}
    curr_score = 0
    for j in bins:
            curr_bin = j
            if j not in person1_bins:
                curr_score = curr_score
            else:
                curr_score = person1_bins[j]
            scores[curr_bin] = curr_score
    result.append(scores)
print(result)

In [ ]:
import matplotlib.pyplot as plt

for dataset in result:
    x = list(dataset.keys())
    y = list(dataset.values())
    plt.plot(x, y)

# Add labels and title
plt.xlabel('Bins -15000 -> 0')
plt.ylabel('Jccard Score')
plt.title('Person 1 Bins vs Score')

# Display the graph
plt.grid(True)
plt.show()

In [ ]:
max_key = max(people_compared, key=people_compared.get)

# Get the maximum value
max_value = people_compared[max_key]

# Return the key-value pair
max_pair = (max_key, max_value)

print(max_pair)

In [ ]:
#RUN THIS ONE
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

unique_numbers = labels

# Count occurrences of each number in each list
count_hypertension = [hypertension_bins.count(num) for num in unique_numbers]
# count_hyperlipidemia = [hyperlipidemia_bins.count(num) for num in unique_numbers]
# count_e_hypertension = [e_hypertension_bins.count(num) for num in unique_numbers]
# count_hypercholesterolema = [hypercholesterolema_bins.count(num) for num in unique_numbers]
# count_diabetes = [diabetes_bins.count(num) for num in unique_numbers]

# Bar width
bar_width = 0.2

# Positions of the bars on the x-axis
r1 = np.arange(len(unique_numbers))
# r2 = [x + bar_width for x in r1]
# r3 = [x + bar_width for x in r2]
# r4 = [x + bar_width for x in r3]
# r5 = [x + bar_width for x in r4]

# Create the plot
plt.figure(figsize=(18, 9))

# Plot each list with different colors
plt.bar(r1, count_hypertension, color='blue', width=bar_width, label='Hypertension')
# plt.bar(r2, count_hyperlipidemia, color='green', width=bar_width, label='272.4')
# plt.bar(r3, count_e_hypertension, color='red', width=bar_width, label='I10')
# plt.bar(r4, count_hypercholesterolema, color='purple', width=bar_width, label='272.0')
# plt.bar(r5, count_diabetes, color='orange', width=bar_width, label='250.00')

# Adding title and labels
plt.title('Pre-Cancer Hypertension Frequency', fontsize = 25)
plt.xlabel('Days Before Cancer Diagnosis (Bins)', fontsize = 20)
plt.ylabel('Frequency of Hypertension Event', fontsize = 20)
plt.xticks([r + bar_width for r in range(len(unique_numbers))], unique_numbers, rotation=90, fontsize=8)

# Adding a legend
plt.legend()

# Display the plot
#plt.show()
plt.savefig("HyperTension_v2.pdf",format="pdf")
#plt.savefig("HyperTension.png")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

unique_numbers = labels

# Count occurrences of each number in each list
count_benign = [benign_bins.count(num) for num in unique_numbers]

# Bar width
bar_width = 0.2

# Positions of the bars on the x-axis
r1 = np.arange(len(unique_numbers))

# Create the plot
plt.figure(figsize=(18, 9))

# Plot each list with different colors
plt.bar(r1, count_benign, color='blue', width=bar_width, label='Benign Neoplasms')

# Adding title and labels
plt.title('Timeline of Benign Neoplasms')
plt.xlabel('Bins')
plt.ylabel('Count')
plt.xticks([r + bar_width for r in range(len(unique_numbers))], unique_numbers, rotation=90, fontsize=8)

# Adding a legend
plt.legend()

# Display the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Flatten the list of sets into a single list of elements
flat_list = [item for subset in matching_codes for item in subset if item is not None]

# Use Counter to count the occurrences of each unique element
counter = Counter(flat_list)

# Filter out items with counts less than or equal to 100
filtered_counter = {item: count for item, count in counter.items() if count > 100}

# Extract unique items and their counts from the filtered data
unique_items = list(filtered_counter.keys())
counts = list(filtered_counter.values())

# Create the bar graph
plt.figure(figsize=(30, 10))  # Adjust the figure size as needed
bars = plt.bar(unique_items, counts, color='skyblue')

# Rotate x-axis labels for better readability
plt.xticks(rotation=90, ha='right', fontsize=20)

# Add labels and title
plt.xlabel('ICD Codes', fontsize=25)
plt.ylabel('Count', fontsize=25)
plt.title('Frequency of Matching ICD Codes (Count > 100)', fontsize=30)

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
max_scored_bins = []
for score in jscore_to_bin_dict:
    if score >= 0.1:
        max_scored_bins.append(jscore_to_bin_dict[score])
print(max_scored_bins)

In [ ]:
means = []
medians = []

In [ ]:
time_diffs = []
for item in max_scored_bins:
    numbers = []
    for i in item:
        split_numbers = [int(num) for num in i.split(',')]
        # Append the converted numbers to the list
        numbers.extend(split_numbers)
    diff = numbers[0] - numbers[2]
    time_diffs.append(diff)
absolute_time = [abs(num) for num in time_diffs]
print(absolute_time)
tm = st.mean(absolute_time) / 365.25
tme = st.median(absolute_time) / 365.25
means.append(tm)
medians.append(tme)

In [ ]:
print(means)
print(medians)
print(st.mean(means))
print(st.mean(medians))

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

counter = Counter(max_scored_bins)

# Extract unique items and their counts from the Counter
unique_items = list(counter.keys())
counts = list(counter.values())

# Create the bar graph
plt.figure(figsize=(14, 20))  # Increase the figure size for readability
bars = plt.barh([str(item) for item in unique_items], counts, color='skyblue')

# Add labels and title
plt.xlabel('Bins Compared', fontsize=20)
plt.ylabel('Unique Items', fontsize=20)
plt.title('Bins with over 10% similarity Count', fontsize=25)

# Adjust y-label font size for readability
plt.yticks(fontsize=10)

# Adjust layout to make room for the labels
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
new_graph_matrix = []

In [ ]:
for i in multiple_links:
    if len(i) >= 10:
        new_graph_matrix.append(i)

In [ ]:
import matplotlib.pyplot as plt

def plot_matrix(matrix):
    for sublist in matrix:
        plt.plot(sublist)

    plt.xlabel('Comparisons based on Person 1 timeline')
    plt.ylabel('Average Jaccard Scores')
    plt.title('Jaccard Scores of people with >= 10 Comparisons')
    plt.grid(True)
    plt.show()

plot_matrix(new_graph_matrix)

In [ ]:
twentyormore = []

In [ ]:
for i in multiple_links:
    if len(i) >= 20:
        twentyormore.append(i)

In [ ]:
import matplotlib.pyplot as plt

def plot_matrix(matrix):
    for sublist in matrix:
        plt.plot(sublist)

    plt.xlabel('Comparisons based on Person 1 timeline')
    plt.ylabel('Average Jaccard Scores')
    plt.title('Jaccard Scores of people with >= 20 Comparisons')
    plt.grid(True)
    plt.show()

plot_matrix(twentyormore)

In [ ]:
point1 = []

In [ ]:
for i in multiple_links:
    if max(i) >= 0.1:
        point1.append(i)
    elif min(i) <= -0.1:
        point1.append(i)

In [ ]:
import matplotlib.pyplot as plt

def plot_matrix(matrix):
    for sublist in matrix:
        plt.plot(sublist)

    plt.xlabel('Comparisons based on Person 1 timeline')
    plt.ylabel('Average Jaccard Scores')
    plt.title('Jaccard Scores of people with scores >= 0.1/-0.1')
    plt.grid(True)
    plt.show()

plot_matrix(point1)

In [ ]:
pointo5 = []

In [ ]:
for i in multiple_links:
    if max(i) >= 0.05:
        pointo5.append(i)
    elif min(i) <= -0.05:
        pointo5.append(i)

In [ ]:
import matplotlib.pyplot as plt

def plot_matrix(matrix):
    for sublist in matrix:
        plt.plot(sublist)

    plt.xlabel('Comparisons based on Person 1 timeline')
    plt.ylabel('Average Jaccard Scores')
    plt.title('Jaccard Scores of people with scores >= 0.05/-0.05')
    plt.grid(True)
    plt.show()

plot_matrix(pointo5)

In [ ]:
# print(indexes2actuallyplot)

In [ ]:
selected_items = []
for index_tuple in indexes2actuallyplot:
    for row, col in index_tuple:
        if 0 <= row < len(bins_compared):
            if 0 <= col < len(bins_compared[row]):
                selected_items.append(bins_compared[row][col])
print(selected_items)

print()

In [ ]:
all_bins_used = []

In [ ]:
all_bins_used.append(selected_items)
# print(all_bins_used)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def density_plot_matrix(matrix):
    # Create a dictionary to store the count of each unique item
    item_counts = {}

    # Count the occurrences of each unique item in the matrix
    for sublist in matrix:
        for item in sublist:
            # Convert the tuple to a string for use as a dictionary key
            item_str = str(item)
            if item_str in item_counts:
                item_counts[item_str] += 1
            else:
                item_counts[item_str] = 1

    # Extract unique items and their counts
    unique_items = list(item_counts.keys())
    counts = [item_counts[item] for item in unique_items]

    # Plot the density plot
    plt.figure(figsize=(26, 30))
    plt.barh(unique_items, counts, color='skyblue')
    plt.xlabel('Count', fontsize=25)
    plt.ylabel('Bin Comparisons', fontsize=25)
    plt.title('All Bins Compared', fontsize=30)
    plt.yticks(ticks=range(len(unique_items)), labels=unique_items, va='center', rotation=0, ha='right', fontsize=10)
    plt.tight_layout()
    plt.show()

density_plot_matrix(all_bins_used)

# Day Diff Distributions

In [ ]:
# Hist distribution of time from first EHR record to cancer

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# groups by unique individuals, and gets the first EHR record from each individual.
first_event_df = binned_individuals.groupby('person_id')['day_diff'].min().reset_index()

sns.histplot(data=first_event_df, x='day_diff', bins=18, kde=True)

plt.xlabel('Days Before Diagnosis')
plt.ylabel('People')
plt.title('Time From First EHR Record to Diagnosis')

plt.show()

In [ ]:
# people who had EHR events binned by day diff

# this groups the data by day_diff, then counts the number of unique person_ids per group.
# results in a dataframe where each row contains a unique day_diff, and a count of unique people who had that day diff.
daydiff_grouped_df = binned_individuals.groupby('day_diff')['person_id'].nunique().reset_index()

daydiff_grouped_df.columns = ['day_diff', 'unique_person_count']

# example to show what dataframe looks like
print(daydiff_grouped_df.head())

# graph bins by day_diff, and shows how many unique people had an EHR event within a certain range of day_diffs
sns.histplot(data=daydiff_grouped_df, x='day_diff', weights='unique_person_count', bins=18, kde=True)

plt.xlabel('Days Before Diagnosis')
plt.ylabel('Number of People w/ EHR Event')
plt.title('EHR Events by Days Before Diagnosis')

plt.show()

# End of graph code

In [ ]:
from statistics import *

max_scores = []
for r in matrix_scores:
    for i in r:
        max_scores.append(i)
avg = mean(max_scores)
average_scores.append(avg)

print(average_scores)

In [ ]:
pairs = []
for _ in range(100):
    peeps = []
    rando1 = new_minmax_data['person_id'].sample().iloc[0]
    rando2 = new_minmax_data['person_id'].sample().iloc[0]
    peeps.append(rando1)
    peeps.append(rando2)
    pairs.append(peeps)
print(pairs)

In [ ]:
for i in pairs:
    A = binned_individuals.loc[(binned_individuals['person_id'] == i[0])]
    B = binned_individuals.loc[(binned_individuals['person_id'] == i[1])]

    similarity_scores = []

    for p in A['bin']:
        bin_scores = []
        for j in B['bin']:
            Asub = set(A.loc[(A['bin'] == p), 'condition_source_value'])
            Bsub = set(B.loc[(B['bin'] == j), 'condition_source_value'])
            C = Asub.intersection(Bsub)
            D = Asub.union(Bsub)
            if len(D) == 0:
                jsim = 0.0
            else:
                jsim = float(len(C))/float(len(D))
            bin_scores.append(jsim)
        similarity_scores.append(bin_scores)

    max_scores = []
    for r in similarity_scores:
        max_scores.append(max(r))
    avg = mean(max_scores)

    this_test = pd.DataFrame({'id': [(i[0], i[1])], 'jscore': avg})

    jaccard_tests = pd.concat([jaccard_tests, this_test])

In [ ]:
jaccard_tests

In [ ]:
max_v = jaccard_tests['jscore'].max()
max_row = jaccard_tests[jaccard_tests['jscore'] == max_v]
print(max_row)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.plot(similarity_matrix)

plt.xticks(range(len(similarity_matrix)))

plt.xlabel("Bins")
plt.ylabel("Jaccard Score")
plt.title("Similarity Plot")

plt.xticks(rotation=45)

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# make boxplots, find coefficient of variation. Order bins based on coefficient of
# variations, then take the smallest variation

In [ ]:
# we have 29543 individuals that we work with

# BELOW: groups each corresponding bin together,
# then grabs the count of how many events fall in that bin (for all individuals)
grouped_counts = binned_individuals.groupby('bin').size()
grouped_counts.head(30)

#BELOW: changes the grouped_counts to be a dataframe, then resets the index to create a separate column "Count"
grouped_cancer_data = grouped_counts.reset_index(name='Count')
#BELOW: grabs each count and divides it by the number of total individuals (grabs average number of events per person in that respective bin)
grouped_cancer_data['Average'] = grouped_cancer_data['Count'] / 29543
grouped_cancer_data.head(30)


plt.scatter(grouped_cancer_data['bin'], grouped_cancer_data['Average'], label='Average')
plt.title('Scatter Plot of Bins')
plt.xlabel('Bin')
plt.ylabel('Avg events/Person')
plt.legend()
plt.show()

In [ ]:
grouped_cancer_data.head()

Using concept_codes in jaccard scores:

In [ ]:
beforeandafter_codes = pd.DataFrame()
shared_data_codes = all_codes['person_id'].isin(new_minmax_data['person_id'])
beforeandafter_codes = all_codes[shared_data_codes]
beforeandafter_codes.head()

In [ ]:
banda_codes_sorted = beforeandafter_codes.sort_values(by='person_id').groupby('person_id', group_keys=False).apply(lambda x: x.sort_values(by='condition_start_datetime'))
banda_codes_sorted = banda_codes_sorted.reset_index(drop=True)
banda_codes_sorted.head()

In [ ]:
# Group new_minmax_data by 'person_id' and select the 'amin' value for each person
amin_mapping = new_minmax_data.groupby('person_id')['amin'].first()

# Create 'day0_relative_time' by mapping 'person_id' to the corresponding 'amin' value
banda_codes_sorted['day0_relative_time'] = banda_codes_sorted['person_id'].map(amin_mapping)

# Calculate 'day_diff' using the mapping and vectorized operations
banda_codes_sorted['day_diff'] = (banda_codes_sorted['condition_start_datetime'] - banda_codes_sorted['day0_relative_time']).dt.days
del banda_codes_sorted['day0_relative_time']
banda_codes_sorted.head()

In [ ]:
# uses vectorized function to filter negative days

neg_days_codes = banda_codes_sorted[banda_codes_sorted['day_diff'] <= 0]

neg_days_codes.head()

In [ ]:
neg_days_codes['bin'] = pd.cut(neg_days_codes['day_diff'],labels=labels,bins=bins)
binned_codes_data = neg_days_codes
binned_codes_data.head(30)

In [ ]:
binned_individuals_codes = binned_codes_data[pd.notna(binned_codes_data['bin'])]
binned_individuals_codes.head(30)

In [ ]:
# codes_rando1 = new_minmax_data['person_id'].sample().iloc[0]
# codes_rando2 = new_minmax_data['person_id'].sample().iloc[0]
# print(codes_rando1)
# print(codes_rando2)
codes_rando1 = 1463039
codes_rando2 = 4403964

In [ ]:
similarity_codes_scores = []

A = binned_individuals_codes.loc[(binned_individuals_codes['person_id'] == codes_rando1)]
B = binned_individuals_codes.loc[(binned_individuals_codes['person_id'] == codes_rando2)]

for i in A['bin']:
    bin_scores = []
    for j in B['bin']:
        Asub = set(A.loc[(A['bin'] == i), 'concept_code'])
        Bsub = set(B.loc[(B['bin'] == j), 'concept_code'])
        C = Asub.intersection(Bsub)
        D = Asub.union(Bsub)
        if len(D) == 0:
            jsim = 0.0
        else:
            jsim = float(len(C))/float(len(D))
        bin_scores.append(jsim)
    similarity_codes_scores.append(bin_scores)

In [ ]:
codes_j_tests = pd.DataFrame(columns = ['id', 'jscore'])

In [ ]:
from statistics import *

max_scores = []
for r in similarity_codes_scores:
    max_scores.append(max(r))
avg = mean(max_scores)

this_test = pd.DataFrame({'id': [(codes_rando1, codes_rando2)], 'jscore': avg})

codes_j_tests = pd.concat([codes_j_tests, this_test])

In [ ]:
print(codes_j_tests)

# Malignant Breast Cancer Control Group

In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd

In [ ]:
# This snippet assumes you run setup first

# This code copies file in your Google Bucket and loads it into a dataframe

# Replace 'test.csv' with THE NAME of the file you're going to download from the bucket (don't delete the quotation marks)
name_of_file_in_bucket = 'control_group_malignant.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
control_group_malignant = pd.read_csv(name_of_file_in_bucket)
control_group_malignant.head()

# 83 x 83 x 83

In [ ]:
# first argument is how many days back our bins should go, the second is the size of each bin
# returns two lists, the first is the cuts of each bin and the second is the name of the respective bins
def bins_n_labels(time_range, bin_size):
    bins = []
    labels = []
    n = -time_range
    while n <= 0:
        bins.append(n)
        label = f'{n},'
        n += bin_size
        label += f'{n}'
        labels.append(label)
    return bins, labels[:-1]

bins, labels = bins_n_labels(15000,100)
# arg 1 is how far back you want to look, arg 2 is the size of each bin you want.
print(bins)
# print(bins)
print(len(bins))
# print(labels)
print(len(labels))
# neg_days['bins'] = pd.cut(neg_days['values'], bins=bins, labels=labels)

In [ ]:
#pull in control group people

import os
import subprocess
import numpy as np
import pandas as pd

# This snippet assumes you run setup first

# This code copies file in your Google Bucket and loads it into a dataframe

# Replace 'test.csv' with THE NAME of the file you're going to download from the bucket (don't delete the quotation marks)
name_of_file_in_bucket = 'control_group_malignant.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
control_dataframe = pd.read_csv(name_of_file_in_bucket)
control_dataframe.head()

print(len(control_dataframe))

In [ ]:
# Bins each event
control_dataframe['bin'] = pd.cut(control_dataframe['day_diff'],labels=labels,bins=bins)
control_dataframe.head()

value_counts = control_dataframe['bin'].value_counts().sort_index().reset_index()
value_counts.head(10)
num_events = value_counts['count'].tolist()
print(num_events)

control_dataframe = control_dataframe[pd.notna(control_dataframe['bin'])]
control_dataframe['concept_code'] = control_dataframe['standard_concept_code']
control_dataframe.head(30)

In [ ]:
import matplotlib.pyplot as plt

counts_df = control_dataframe.groupby('person_id').size().reset_index(name='count')
mean_counts = np.mean(counts_df['count'])
median_counts = np.median(counts_df['count'])
print(f'mean: {mean_counts}')
print(f'median: {median_counts}')

filtered_counts = counts_df[(counts_df["count"] > 100) & (counts_df["count"] < 1200)]
ids = filtered_counts['person_id']
filtered_control = control_dataframe[control_dataframe['person_id'].isin(ids)]
print('Have at least 100 EHRs:')
print(len(filtered_control["person_id"].unique().tolist()))

Q1_pres = filtered_counts['count'].quantile(0.25)
Q3_pres = filtered_counts['count'].quantile(0.75)
IQR_pres = Q3_pres - Q1_pres
print(f'Q1: {Q1_pres}')
print(f'Q3: {Q3_pres}')
print(f'IQR: {IQR_pres}')

cap_pres = 1.5 * IQR_pres + Q3_pres
print(cap_pres)

filtered_counts["count"].plot(kind="density")
plt.xlabel("Counts")
plt.title("Density Plot of Counts")
plt.show()

In [ ]:
# SETUP

import os
import subprocess
import numpy as np
import pandas as pd


# This snippet assumes you run setup first
# Get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')
# List objects in the bucket
# print(subprocess.check_output(f"gsutil ls -r {my_bucket}", shell=True).decode('utf-8'))
# This code copies file in your Google Bucket and loads it into a dataframe

# Replace 'test.csv' with THE NAME of the file you're going to download from the bucket (don't delete the quotation marks)
name_of_file_in_bucket = 'binned_individuals.csv'

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

# print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
binned_individuals = pd.read_csv(name_of_file_in_bucket)
binned_individuals.head()

In [ ]:
counts_ind = binned_individuals.groupby('person_id').size().reset_index(name='count')
filtered_counts = counts_ind[(counts_ind["count"] > 100) & (counts_ind["count"] < 1200)]
ids = filtered_counts['person_id']
filtered_binned = binned_individuals[binned_individuals['person_id'].isin(ids)]
print(len(filtered_binned["person_id"].unique().tolist()))

Q1 = counts_ind['count'].quantile(0.25)
Q3 = counts_ind['count'].quantile(0.75)
IQR = Q3 - Q1
print(f'Q1: {Q1}')
print(f'Q3: {Q3}')
print(f'IQR: {IQR}')
mean_binned = np.mean(counts_ind['count'])

cap = 1.5 * IQR + Q3
print(cap)

In [ ]:
#filtered_control = pre-vivors
counts_df = control_dataframe.groupby('person_id').size().reset_index(name='count')
filtered_counts = counts_df[(counts_df["count"] > 100) & (counts_df["count"] < 1200)]
pre_ids = filtered_counts['person_id']
filtered_control = control_dataframe[control_dataframe['person_id'].isin(pre_ids)]

In [ ]:
print(len(filtered_binned))

In [ ]:
#unique_people = test group, randos with >100 EHRs
total_people = filtered_binned['person_id'].unique().tolist()
num_individuals = 75
unique_people = filtered_binned['person_id'].sample(num_individuals, replace=False).tolist()
excluded = set(unique_people) | set(pre_ids)
total_people = [person for person in total_people if person not in excluded]

In [ ]:
print(len(filtered_binned))

In [ ]:
#matched_people = randos with >100 EHRs
num_individuals = 75
new_sample = filtered_binned[filtered_binned['person_id'].isin(total_people)]
matched_people = new_sample['person_id'].sample(num_individuals, replace=False).tolist()

In [ ]:
print(list(set(pre_ids) & set(matched_people)))

In [ ]:
filtered_control.head()

In [ ]:
#pre-vivors vs matched
from collections import OrderedDict
import statistics as st
import numpy as np
import itertools
import copy
import sys
import pandas as pd

fixed_people = filtered_control['person_id'].drop_duplicates().tolist()

indexes2actuallyplot = []
jscore_to_bin_dict = {}
matching_codes = []
people_compared = {}
people_bins_avgScores = {}
averages_with_bins = {}
split_avg = {}

for rando1 in fixed_people:
    for rando2 in matched_people:
        person2_df = filtered_control[filtered_control['person_id'] == rando1]
        person1_df = filtered_binned[filtered_binned['person_id'] == rando2]

        person1_bin_set = set(person1_df['bin'])
        person2_bin_set = set(person2_df['bin'])

        bin_pairs = np.array([(bin1, bin2) for bin1 in person1_bin_set for bin2 in person2_bin_set], dtype=object)

        Asub_list = {bin1: set(person1_df.loc[person1_df['bin'] == bin1, 'concept_code']) for bin1 in person1_bin_set}
        Bsub_list = {bin2: set(person2_df.loc[person2_df['bin'] == bin2, 'concept_code']) for bin2 in person2_bin_set}

        similarity_scores = np.array([
            len(Asub_list[bin1] & Bsub_list[bin2]) / len(Asub_list[bin1] | Bsub_list[bin2]) if len(Asub_list[bin1] | Bsub_list[bin2]) > 0 else 0.0
            for bin1, bin2 in bin_pairs
        ])

        similarity_matrix = similarity_scores.reshape(len(person1_bin_set), len(person2_bin_set))
        mask = np.ones(similarity_matrix.shape, dtype=bool)
        max_dict_ordered = OrderedDict()
        excluded_bins_to_raw_score = OrderedDict()

        while np.any(mask):
            masked_matrix = np.where(mask, similarity_matrix, -np.inf)
            max_index = np.argmax(masked_matrix)
            max_row, max_col = np.unravel_index(max_index, similarity_matrix.shape)
            max_score = similarity_matrix[max_row, max_col]

            tuple_key = tuple(bin_pairs[max_row * len(person2_bin_set) + max_col])
            max_dict_ordered[tuple_key] = max_score
            excluded_bins_to_raw_score.setdefault(tuple_key, []).append(max_score)

            mask[max_row, :] = False
            mask[:, max_col] = False

        sorted_ordered_dict = OrderedDict(sorted(max_dict_ordered.items(), key=lambda x: x[1], reverse=True))
        indexes2actuallyplot.append(list(sorted_ordered_dict.keys()))
        averages_to_actually_graph = [st.mean(sorted_ordered_dict.values())]

        sorted_bins_to_max_score_dict = OrderedDict(sorted(excluded_bins_to_raw_score.items(), key=lambda x: max(x[1]), reverse=True))
        while sorted_bins_to_max_score_dict:
            bins_from_matrix = list(sorted_bins_to_max_score_dict.keys())
            maxes_average = st.mean([val for values in sorted_bins_to_max_score_dict.values() for val in values])
            averages_with_bins.setdefault((rando1, rando2), []).append({bins_from_matrix[-1]: maxes_average})
            sorted_bins_to_max_score_dict.popitem()

        person1_best_scores = {key[0]: value[0] for key, value in excluded_bins_to_raw_score.items()}
        person2_best_scores = {key[1]: value[0] for key, value in excluded_bins_to_raw_score.items()}

        sorted_by_bins_person1 = OrderedDict(sorted(person1_best_scores.items()))
        sorted_by_bins_person2 = OrderedDict(sorted(person2_best_scores.items()))

        if (rando2, rando1) not in split_avg:
            split_avg[(rando2, rando1)] = []

        while sorted_by_bins_person2:
            closest_to_diagnosis_p2 = next(iter(sorted_by_bins_person2))
            curr_average = st.mean(sorted_by_bins_person2.values())
            split_avg[(rando2, rando1)].append({closest_to_diagnosis_p2: curr_average})
            del sorted_by_bins_person2[closest_to_diagnosis_p2]

In [ ]:
print(list(set(fixed_people) & set(matched_people)))

In [ ]:
result1 = []
comparison_ids = [] # This is storing tuples
# the key is the person, the value is the dictionary of bins to scores
for key, value in split_avg.items():
    person2_bins = {}
    for val in value:
        for k, v in val.items():
            bin = int(k.split(',')[0])
            person2_bins[bin] = v
    scores = {}
    score_list = []
    curr_score = 0
    for j in bins:
            curr_bin = j
            if j not in person2_bins:
                curr_score = curr_score
            else:
                curr_score = person2_bins[j]
            scores[curr_bin] = curr_score
            score_list.append(curr_score)
    max_score = 0
    for item in score_list:
        if item > max_score:
            max_score = item
    if max_score != 1.0:
        result1.append(scores)
        comparison_ids.append(key)

In [ ]:
id_to_list_of_dictionaries = {}
for person in fixed_people:
    id_to_list_of_dictionaries[person] = []
    for group_index in range(len(result1)):
        if comparison_ids[group_index][1] == person:
            id_to_list_of_dictionaries[person].append(result1[group_index])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np  # Make sure this is imported

plt.figure(figsize=(20,14))

comparison_functions = {}
apex_x_values = {}  # Optional: store apex x-values by ID
total_windel = {}

for previd, group in id_to_list_of_dictionaries.items():
    total_windel[previd] = []
    i = -1
    for dataset in group:

        i += 1
        x = np.array(list(dataset.keys()))
        y = np.array(list(dataset.values()))

        # Find the cutoff to ignore leading zero values
        index = 0
        while index < len(y) and y[index] == 0:
            index += 1
        y_cutoff = y[index:]
        x_cutoff = x[index:]

        if x_cutoff.size == 0:
            continue

        # Fit a quadratic model
        coefficients = np.polyfit(x_cutoff, y_cutoff, 2)  # coefficients = [a, b, c]
        a, b = coefficients[0], coefficients[1]
        comparison_functions[comparison_ids[i]] = coefficients

        # Create model and predictions
        quadratic_model = np.poly1d(coefficients)
        x_fit = np.linspace(x.min(), x.max(), 100)
        y_fit = quadratic_model(x_fit)

        # Only plot lines with a positive quadratic coefficient
#         if a < -5.8e-08:
        plt.plot(x_fit, y_fit)

        # Calculate apex x-value: x = -b / (2a)
        x_apex = -b / (2 * a)
        apex_x_values[comparison_ids[i]] = x_apex
        total_windel[previd].append(x_apex)
    #         print(f"Apex for {comparison_ids[i]}: x = {x_apex:.2f}")

plt.gca().invert_xaxis()
plt.xlabel('Days Before Cancer Diagnosis (Bins)', fontsize=20)
plt.ylabel('Jaccard Score', fontsize=20)
plt.title('Pre-vivors vs Matched', fontsize=25)
plt.xlim(0, -3000)
plt.ylim(-.05, .25)
plt.grid(True)
plt.show()

In [ ]:
best_windel = {}

for identification, w_scores in total_windel.items():
    w_scores = [t for t in w_scores if t < 0]
    best_windel[identification] = max(w_scores)

df = pd.DataFrame(list(best_windel.items()), columns=['id', 'best_score'])
print(df)

In [ ]:
#plot previvors windel scores on box plot
import matplotlib.pyplot as plt

df['best_score'].plot(kind='box')
plt.ylabel('Days before Cancer Diagnosis')
plt.title('Pre-Vivor Best Windel Scores')
plt.show()

In [ ]:
#randos vs matched
from collections import OrderedDict
import statistics as st
import numpy as np
import itertools
import copy
import sys
import pandas as pd

indexes2actuallyplot = []
jscore_to_bin_dict = {}
matching_codes = []
people_compared = {}
people_bins_avgScores = {}
averages_with_bins = {}
split_avg_test = {}

for rando1 in unique_people:
    for rando2 in matched_people:
        person1_df = filtered_binned[filtered_binned['person_id'] == rando2]
        person2_df = filtered_binned[filtered_binned['person_id'] == rando1]

        person1_bin_set = set(person1_df['bin'])
        person2_bin_set = set(person2_df['bin'])

        bin_pairs = np.array([(bin1, bin2) for bin1 in person1_bin_set for bin2 in person2_bin_set], dtype=object)

        Asub_list = {bin1: set(person1_df.loc[person1_df['bin'] == bin1, 'concept_code']) for bin1 in person1_bin_set}
        Bsub_list = {bin2: set(person2_df.loc[person2_df['bin'] == bin2, 'concept_code']) for bin2 in person2_bin_set}
        similarity_scores = np.array([
            len(Asub_list[bin1] & Bsub_list[bin2]) / len(Asub_list[bin1] | Bsub_list[bin2]) if len(Asub_list[bin1] | Bsub_list[bin2]) > 0 else 0.0
            for bin1, bin2 in bin_pairs
        ])

        similarity_matrix = similarity_scores.reshape(len(person1_bin_set), len(person2_bin_set))
        mask = np.ones(similarity_matrix.shape, dtype=bool)
        max_dict_ordered = OrderedDict()
        excluded_bins_to_raw_score = OrderedDict()

        while np.any(mask):
            masked_matrix = np.where(mask, similarity_matrix, -np.inf)
            max_index = np.argmax(masked_matrix)
            max_row, max_col = np.unravel_index(max_index, similarity_matrix.shape)
            max_score = similarity_matrix[max_row, max_col]

            tuple_key = tuple(bin_pairs[max_row * len(person2_bin_set) + max_col])
            max_dict_ordered[tuple_key] = max_score
            excluded_bins_to_raw_score.setdefault(tuple_key, []).append(max_score)

            mask[max_row, :] = False
            mask[:, max_col] = False
        sorted_ordered_dict = OrderedDict(sorted(max_dict_ordered.items(), key=lambda x: x[1], reverse=True))
        indexes2actuallyplot.append(list(sorted_ordered_dict.keys()))
        averages_to_actually_graph = [st.mean(sorted_ordered_dict.values())]

        sorted_bins_to_max_score_dict = OrderedDict(sorted(excluded_bins_to_raw_score.items(), key=lambda x: max(x[1]), reverse=True))
        while sorted_bins_to_max_score_dict:
            bins_from_matrix = list(sorted_bins_to_max_score_dict.keys())
            maxes_average = st.mean([val for values in sorted_bins_to_max_score_dict.values() for val in values])
            averages_with_bins.setdefault((rando1, rando2), []).append({bins_from_matrix[-1]: maxes_average})
            sorted_bins_to_max_score_dict.popitem()

        person1_best_scores = {key[0]: value[0] for key, value in excluded_bins_to_raw_score.items()}
        person2_best_scores = {key[1]: value[0] for key, value in excluded_bins_to_raw_score.items()}

        sorted_by_bins_person1 = OrderedDict(sorted(person1_best_scores.items()))
        sorted_by_bins_person2 = OrderedDict(sorted(person2_best_scores.items()))

        if (rando2, rando1) not in split_avg_test:
            split_avg_test[(rando2, rando1)] = []

        while sorted_by_bins_person2:
            closest_to_diagnosis_p2 = next(iter(sorted_by_bins_person2))
            curr_average = st.mean(sorted_by_bins_person2.values())
            split_avg_test[(rando2, rando1)].append({closest_to_diagnosis_p2: curr_average})
            del sorted_by_bins_person2[closest_to_diagnosis_p2]

In [ ]:
result2 = []
comparison_ids_test = [] # This is storing tuples
# the key is the person, the person is the dictionary of bins to scores
for key, value in split_avg_test.items():
    person2_bins = {}
    for val in value:
        for k, v in val.items():
            bin = int(k.split(',')[0])
            person2_bins[bin] = v
    scores = {}
    score_list = []
    curr_score = 0
    for j in bins:
        curr_bin = j
        if j not in person2_bins:
            curr_score = curr_score
        else:
            curr_score = person2_bins[j]
        scores[curr_bin] = curr_score
        score_list.append(curr_score)
    max_score = 0
    for item in score_list:
        if item > max_score:
            max_score = item
    if max_score != 1.0:
        result2.append(scores)
        comparison_ids_test.append(key)

In [ ]:
test_id_to_list_of_dictionaries = {}
for test_person in unique_people:
    test_id_to_list_of_dictionaries[test_person] = []
    for test_index in range(len(result2)):
        if comparison_ids_test[test_index][1] == test_person:
            test_id_to_list_of_dictionaries[test_person].append(result2[test_index])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np  # Make sure this is imported

plt.figure(figsize=(20,14))

comparison_functions_test = {}
test_x_values = {}  # Optional: store apex x-values by ID
test_windel = {}

for test_id, test_group in test_id_to_list_of_dictionaries.items():
    test_windel[test_id] = []
    i_test = -1
    for dataset in test_group:
        i_test += 1
        x_test = np.array(list(dataset.keys()))
        y_test = np.array(list(dataset.values()))

        # Find the cutoff to ignore leading zero values
        index_test = 0
        while index_test < len(y_test) and y_test[index_test] == 0:
            index_test += 1
        y_cutoff_test = y_test[index_test:]
        x_cutoff_test = x_test[index_test:]

        if x_cutoff_test.size == 0:
            continue

        # Fit a quadratic model
        coefficients_test = np.polyfit(x_cutoff_test, y_cutoff_test, 2)  # coefficients = [a, b, c]
        a_test, b_test = coefficients_test[0], coefficients_test[1]
        comparison_functions_test[comparison_ids_test[i_test]] = coefficients_test

        # Create model and predictions
        quadratic_model_test = np.poly1d(coefficients_test)
        x_fit_test = np.linspace(x_test.min(), x_test.max(), 100)
        y_fit_test = quadratic_model_test(x_fit_test)

        # Only plot lines with a positive quadratic coefficient
#         if a_test < -5.8e-08:
        plt.plot(x_fit_test, y_fit_test)

        # Calculate apex x-value: x = -b / (2a)
        x_apex_test = -b_test / (2 * a_test)
        test_x_values[comparison_ids_test[i_test]] = x_apex_test
        test_windel[test_id].append(x_apex_test)
    #         print(f"Apex for {comparison_ids[i]}: x = {x_apex:.2f}")

plt.gca().invert_xaxis()
plt.xlabel('Days Before Cancer Diagnosis (Bins)', fontsize=20)
plt.ylabel('Jaccard Score', fontsize=20)
plt.title('Test vs Matched', fontsize=25)
plt.xlim(0, -3000)
plt.ylim(-.05, .25)
plt.grid(True)
plt.show()

In [ ]:
test_best_windel = {}

for identification, w_scores in test_windel.items():
    w_scores = [t for t in w_scores if t < 0]
    test_best_windel[identification] = max(w_scores)

test_df = pd.DataFrame(list(test_best_windel.items()), columns=['id', 'test_best_score'])
print(test_df)

In [ ]:
#plot test windel scores on box plot
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 10))
test_df['test_best_score'].plot(kind='box')
plt.ylabel('Days before Cancer Diagnosis')
plt.title('Test Best Windel Scores')

plt.show()

In [ ]:
import matplotlib.pyplot as plt

test_scores = test_df['test_best_score'].tolist()
previvor_scores = df['best_score'].tolist()

# Assuming apex_x_list and test_x_list are lists of numbers (e.g., apex x-values)
data = [previvor_scores, test_scores]

plt.figure(figsize=(10, 12))
plt.boxplot(data, labels=["Pre-vivors", "Test"])
plt.ylabel("Days Before Cancer Diagnosis")
plt.title("Comparison of Best WINDEL Scores")
plt.grid(True)
plt.savefig("best_windel_comparison.pdf", format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
from scipy.stats import ttest_ind
t_stat, p_value = ttest_ind(previvor_scores, test_scores)
print(t_stat)
print(p_value)

# With Average Windel Score (not reliable)

In [ ]:
import numpy as np

pre_avg_windel = {}

for identification, w_scores in total_windel.items():
    w_scores = [t for t in w_scores if t < 0]
    pre_avg_windel[identification] = np.mean(w_scores)

avg_df = pd.DataFrame(list(pre_avg_windel.items()), columns=['id', 'avg_score'])
print(avg_df)

In [ ]:
test_avg_windel = {}

for identification, w_scores in test_windel.items():
    w_scores = [t for t in w_scores if t < 0]
    test_avg_windel[identification] = np.mean(w_scores)

avg_test_df = pd.DataFrame(list(test_avg_windel.items()), columns=['id', 'test_avg_score'])
print(avg_test_df)

In [ ]:
avg_test_scores = avg_test_df['test_avg_score'].tolist()
previvor_avg_scores = avg_df['avg_score'].tolist()

# Assuming apex_x_list and test_x_list are lists of numbers (e.g., apex x-values)
data = [previvor_avg_scores, avg_test_scores]

plt.figure(figsize=(10, 6))
plt.boxplot(data, labels=["Pre-vivors", "Test Group"])
plt.ylabel("Days Before Cancer Diagnosis")
plt.title("Comparison of Average WINDEL Scores")
plt.grid(True)
plt.show()

In [ ]:
from scipy.stats import ttest_ind
t_stat, p_value = ttest_ind(previvor_avg_scores, avg_test_scores)
print(t_stat)
print(p_value)

# Individual Variation

In [ ]:
#graphing individual variation for random cancer test group

fig, axs = plt.subplots(len(test_windel), 1, figsize=(20, 120), constrained_layout=True)

for ax, (key, values) in zip(axs, test_windel.items()):
#     values = [t for t in values if t < 0]
    ax.plot(values, linestyle='-', label=f"ID {key}")  # remove markers for clarity
    ax.set_title(f"Variation of Scores for ID {key}")
    ax.set_xlabel("Index")
    ax.set_ylabel("Value")
    ax.legend()
    ax.grid(True)

plt.show()

In [ ]:
variation_cancer_test = {}

for patient_id, windel_scores in test_windel.items():
    variation = []
    windel_scores = [t for t in windel_scores if t < 0]
    variation.append(max(windel_scores))
    variation.append(np.mean(windel_scores))
    variation.append(np.median(windel_scores))
    variation_cancer_test[patient_id] = variation

variation_df = pd.DataFrame.from_dict(variation_cancer_test, orient='index', columns=['best score', 'mean', 'median']).reset_index().rename(columns={'index': 'id'})
print(variation_df)

In [ ]:
#graphing individual variation for previvor test group

fig, axs = plt.subplots(len(total_windel), 1, figsize=(20, 120), constrained_layout=True)

for ax, (key, values) in zip(axs, total_windel.items()):
#     values = [t for t in values if t < 0]
    ax.plot(values, linestyle='-', label=f"ID {key}")
    ax.set_title(f"Variation of WINDEL Scores for ID {key}")
    ax.set_xlabel("Index")
    ax.set_ylabel("WINDEL Score")
    ax.legend()
    ax.grid(True)

plt.show()

In [ ]:
variation_previvors_test = {}

for pv_patient_id, pv_windel_scores in total_windel.items():
    pv_variation = []
    pv_windel_scores = [t for t in pv_windel_scores if t < 0]
    pv_variation.append(max(pv_windel_scores))
    pv_variation.append(np.mean(pv_windel_scores))
    pv_variation.append(np.median(pv_windel_scores))
    variation_previvors_test[pv_patient_id] = pv_variation

pv_variation_df = pd.DataFrame.from_dict(variation_previvors_test, orient='index', columns=['best score', 'mean', 'median']).reset_index().rename(columns={'index': 'id'})
print(pv_variation_df)

# Total Variation

In [ ]:
# print(df) #previvor df
previvor_sorted = df.sort_values(by='best_score', ascending=False).reset_index()
print(previvor_sorted)

In [ ]:
# print(test_df) #random cancer df
cancer_test_sorted = test_df.sort_values(by='test_best_score', ascending=False).reset_index()
print(cancer_test_sorted)

In [ ]:
import seaborn as sns
sns.kdeplot(pv_variation_df['best score'], fill=True, alpha=0.5, label='Previvors')
sns.kdeplot(variation_df['best score'], fill=True, alpha=0.5, label='Cancer Test')

plt.xlabel('WINDEL Score')
plt.ylabel('Density')
plt.title('Best Scores Comparison: Previvors v Cancer Test')
plt.legend()
plt.show()

In [ ]:
sns.kdeplot(pv_variation_df['mean'], fill=True, alpha=0.5, label='Previvors')
sns.kdeplot(variation_df['mean'], fill=True, alpha=0.5, label='Cancer Test')

plt.xlabel('WINDEL Score')
plt.ylabel('Density')
plt.title('Mean Comparison: Previvors v Cancer Test')
plt.legend()
plt.show()

In [ ]:
sns.kdeplot(pv_variation_df['median'], fill=True, alpha=0.5, label='Previvors')
sns.kdeplot(variation_df['median'], fill=True, alpha=0.5, label='Cancer Test')

plt.xlabel('WINDEL Score')
plt.ylabel('Density')
plt.title('Median Comparison: Previvors v Cancer Test')
plt.legend()
plt.show()

# Non-Cancer Cohort

In [ ]:
# # Required specs to run this block: 16 CPUs 60 Gigs of RAM
import os
import pandas as pd

dataset = %env WORKSPACE_CDR
dataset

query = f"""
SELECT
    person_id,
    condition_start_datetime,
    condition_source_value,
    source_concept_name as icd_description,
    concept_code,
    c.concept_id
FROM
    `{dataset}.ds_condition_occurrence` co
LEFT JOIN
    `{dataset}.concept` c ON co.condition_concept_id = c.concept_id
WHERE co.PERSON_ID IN
        (
            SELECT
                distinct person_id
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person
            WHERE
                cb_search_person.person_id IN

                (
                    SELECT
                        person_id
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p
                )
        )
        AND NOT (
            (UPPER(source_concept_name) LIKE '%MALIGNANT%' AND UPPER(source_concept_name) LIKE '%NEOPLASM%')
            AND (condition_source_value LIKE 'C%' OR
            condition_source_value LIKE '14%' OR
            condition_source_value LIKE '15%' OR
            condition_source_value LIKE '16%' OR
            condition_source_value LIKE '17%' OR
            condition_source_value LIKE '18%' OR
            condition_source_value LIKE '19%' OR
            condition_source_value LIKE '20%' OR
            condition_source_value LIKE '21%' OR
            condition_source_value LIKE '22%' OR
            condition_source_value LIKE '23%')
        )
"""

all_non_cancer = pd.read_gbq(query,
                        dialect = "standard",
                        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
                        progress_bar_type="tqdm_notebook")

all_non_cancer.head()

In [ ]:
test_query = f"""
SELECT
    concept_id,
    concept_name,
    vocabulary_id
FROM `{dataset}.concept`
WHERE LOWER(vocabulary_id) LIKE '%gender%';
"""

test = pd.read_gbq(test_query,
                   dialect = "standard",
                   use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
                   progress_bar_type="tqdm_notebook")

test

In [ ]:
non_cancer_sorted = all_non_cancer.sort_values(by='person_id').groupby('person_id', group_keys=False).apply(lambda x: x.sort_values(by='condition_start_datetime'))
non_cancer_sorted = non_cancer_sorted.reset_index(drop=True)
non_cancer_sorted.head()

In [ ]:
age_query = f"""
SELECT
    person_id,
    birth_datetime
FROM
    `{dataset}.person`
"""

all_ages = pd.read_gbq(age_query,
                        dialect = "standard",
                        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
                        progress_bar_type="tqdm_notebook")

all_ages.head()